# 10. CRIM — 학습 · Branch Ablation · 평가

TPC-H / TPC-DS 로만 학습하고 BIRD 4개 DB 에서 평가한다. 세 개의 branch 조합 7가지를 같은 seed·같은 데이터로 학습하며,
**`stat+char+ov` (전체 branch) 결과가 논문의 제안모델(CRIM) 수치**이다.

| branch | 입력 | 설명 |
|---|---|---|
| `stat` | 컬럼별 18차원 | 고유값 비율, 빈도 분포, 수치 통계 등 |
| `char` | 컬럼별 18차원 | 숫자/영문/구분자 비율, ID·코드·날짜 패턴 등 문자 형태 |
| `ov`   | 컬럼쌍 26차원 | 값 겹침(Jaccard, containment, 방향성 등) 20 + 쌍 통계 6 |

- 입력: `data/tpc/{sf1,tpcds,train}`, `data/tpc/ground_truth_tpc.json` (`scripts/prepare_tpc.py` 로 생성), `workspace/dataset_mini/bird_data_tables`
- 출력: `workspace/ablation_ranking_metrics_0517_v34_final_ablation[_mask].json`
- 재현성: `apply_seed` 와 결정론적 DataLoader 로 같은 하드웨어에서 같은 가중치를 학습한다.

1. 경로정의 및 필수 라이브러리 설치

## TPC-only variant
This notebook is based on the 0329 version and was modified to build the training dataset from TPC-only sources.
- Excludes training CSV files whose filenames start with `spd_`.
- Uses only `sf1` and `tpcds` groups from `result/ground_truth.json`.
- Keeps the rest of the 0329 modeling pipeline unchanged.


## SPIDER text-lite hybrid
This version keeps TPC as the main training source and adds only a small amount of SPIDER FK positives when both columns look text-like.
- Main cache/negative pool remains TPC-only.
- SPIDER is used only for text-like positive FK pairs.
- Added ratio is controlled by `SPIDER_TEXT_RATIO` (default 10% of main positive count).


## Improved 0329 variant
This version keeps the 0329 TPC-only setting but applies lightweight performance improvements.
- Feature extraction: Task2 expanded from 15 to 24 text-shape features.
- Overlap: expanded from 6/4 style to 10 overlap features.
- Model: simpler OV-gated 3-branch classifier using only diff, mul, ov inputs.
- Training/inference: stronger focal loss, slightly higher LR, stricter inference threshold and overlap floor.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# v34_final_ablation: best protocol로 7 branch ablation
# - POS:NEG = 1:2 (v34_final 동일)
# - Loss: BCE (Focal Loss는 SEED=42에서 효과 없음 확인 → BCE 사용)
# - SWA: epoch 30~150 weight averaging — v34_final과 동일 protocol
# - SEED = 42 (default — no test-based selection)
# - random_split: manual_seed(SEED) — explicit reproducibility
#
# ★ v34_final_ablation_deterministic 패치 (논문 재현성 확보용) ★
# ──────────────────────────────────────────────────────────────────
# 배경:
#   - 같은 SEED로 학습해도 별도 프로세스/세션에서 가중치가 미세하게 달라짐
#     (CUDA atomicAdd 부동소수점 누적 순서, cuDNN 알고리즘 선택,
#      DataLoader worker 의 비결정적 셔플 등이 원인)
#   - 결과: superhero 처럼 borderline 후보가 많은 DB 에서 FP 가 흔들림
#     → 같은 stat+char+ov "Full" 인데도 main 노트북 vs ablation 노트북
#       결과가 2 FP 정도 다르게 나옴 (F1: 0.3226 vs 0.3333)
#
# 해결:
#   apply_seed() 내부에 cuDNN/cuBLAS/PyTorch 글로벌 결정론 플래그 추가.
#   이렇게 하면 같은 하드웨어·드라이버에서 매번 같은 가중치 학습 보장.
#   학습 속도가 10~20% 느려지지만 paper-level 재현성이 더 중요.
# ──────────────────────────────────────────────────────────────────
import random, numpy as np, torch
import os

SEED        = 42
FOCAL_ALPHA = 0.5   # (사용 안 됨 — BCE wrapper)
FOCAL_GAMMA = 2.0   # (사용 안 됨)

def apply_seed(seed):
    """
    완전 결정론적 학습을 위한 seed 적용.
    매 ablation variant 학습 직전에 호출해야 동일한 가중치 학습이 보장됨.
    """
    if seed is None:
        return

    # ── 환경변수 ──────────────────────────────────────────────────
    # PYTHONHASHSEED : dict / set 순서를 결정론적으로 (Python 3.3+)
    # CUBLAS_WORKSPACE_CONFIG : cuBLAS matmul 결정론 (CUDA 10.2+)
    os.environ['PYTHONHASHSEED']         = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    # ── 표준 RNG ──────────────────────────────────────────────────
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # ── cuDNN 결정론 ──────────────────────────────────────────────
    # benchmark=False : 입력 shape 별 최적 알고리즘 자동선택 비활성
    #                    (선택 자체가 비결정적이라 변동 야기)
    # deterministic=True : 결정론적 알고리즘만 사용
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    # ── PyTorch 1.8+ 전역 결정론 ─────────────────────────────────
    # warn_only=True : 결정론 op 가 없는 경우 에러 대신 경고
    #                  (scatter_add 등 일부 op 는 결정론 미지원이라
    #                   warn_only 없으면 RuntimeError 발생 가능)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except AttributeError:
        # PyTorch < 1.8 환경 호환
        pass

print(f'🎯 v34_final_ablation: SEED={SEED}, BCE + SWA per variant (deterministic mode)')

# # [cell1]
# # 1. Google Drive 마운트
# from google.colab import drive
# drive.mount('/content/drive')

# 2. 필수 라이브러리 설치 (필요시 활성화)
# !pip install torch pandas numpy sentence-transformers scikit-learn imbalanced-learn datasketch

post_fix ="0517_v34_final_ablation"

# 3. 경로 설정
base_path = '../workspace/'
dataset_path = '../data/tpc/train' # os.path.join(base_path, 'dataset/tpcds/train')
# v13: 데이터 소스별 분리 경로
TPC_H_PATH  = '../data/tpc/sf1'
TPC_DS_PATH = '../data/tpc/tpcds'
SPIDER_PATH = '../data/tpc/spd_'
model_save_path = os.path.join(base_path, f'multi_task_matching_model_{post_fix}.pth')
ground_truth_path =  '../data/tpc/ground_truth_tpc.json'  # v13: TPC-H+TPC-DS+SPIDER 통합
augmented_data_path = os.path.join(base_path, f'augmented_train_data_{post_fix}.pkl')
cache_dir = os.path.join(base_path, 'cache')
features_cache_path = os.path.join(cache_dir, f'tpcds_features_X_{post_fix}.npy')
metadata_cache_path = os.path.join(cache_dir, f'tpcds_metadata_{post_fix}.pkl')
scaler_path = os.path.join(base_path, f'feature_scaler_{post_fix}.pkl')
ov_scaler_path = os.path.join(base_path, f'feature_scaler_ov_{post_fix}.pkl')
clip_bounds_path = os.path.join(base_path, f'clip_bounds_{post_fix}.pkl')

# 3. 경로 설정
# base_path = '/content/drive/MyDrive/crim/0315/'
# dataset_path = os.path.join(base_path, 'dataset/train')
# ground_truth_path = os.path.join(base_path, 'ground_truth.json')
# model_save_path = os.path.join(base_path, 'multi_task_matching_model_0326.pth')
# augmented_data_path = os.path.join(base_path, 'augmented_train_data_0326.pkl')
# cache_dir = os.path.join(base_path, 'cache')
# features_cache_path = os.path.join(cache_dir, 'tpcds_features_X_0326.npy')
# metadata_cache_path = os.path.join(cache_dir, 'tpcds_metadata_0326.pkl')
# scaler_path = os.path.join(base_path, 'feature_scaler_0326.pkl')
# ov_scaler_path = os.path.join(base_path, 'feature_scaler_ov_0326.pkl')

# 작업 디렉토리 이동
os.chdir(base_path)
print(f"Current Working Directory: {os.getcwd()}")
print(f"Dataset Path: {dataset_path}")


In [ ]:
# [v34_final_ablation] v33_trap5 ckpt 100% 재사용 (1:2 데이터 동일)
import shutil
import os

src_dir = os.path.join(base_path, 'ckpt_0517_v33_trap5')
dst_dir = os.path.join(base_path, f'ckpt_{post_fix}')

if not os.path.exists(src_dir):
    print(f'⚠ v33_trap5 ckpt 없음 ({src_dir})')
elif os.path.exists(dst_dir) and len(os.listdir(dst_dir)) > 0:
    print(f'📦 이미 ckpt 존재 ({dst_dir}) — 스킵 (파일 수: {len(os.listdir(dst_dir))})')
else:
    os.makedirs(dst_dir, exist_ok=True)
    copied = 0
    for fname in os.listdir(src_dir):
        if '_0517_v33_trap5.pkl' in fname:
            new_name = fname.replace('_0517_v33_trap5.pkl', f'_{post_fix}.pkl')
            shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, new_name))
            copied += 1
            print(f'  📋 {fname} → {new_name}')
    print(f'\n✅ ckpt {copied}개 복사 완료')


#### 2. 데이터 로드 및 특성 추출 (Feature Extraction)

##### Task 1: 수치 통계 특성
<small>**Sherlock 기반 27개 피처 | 수치 통계 **</small>

| 범주 | 피처 |
|------|------|
| 기본 통계 | mean, std, min_val, max_val, skewness, kurtosis, sum_val, range_val, IQR, coeff_var |
| 분위수 | q0, q25, q50, q75, q100 |
| 카운트 | total_count, n_distinct, uniqueness_ratio, n_null, null_ratio, is_integer_ratio |
| 길이 특성 | mean_wl, std_wl, min_wl, max_wl, numeric_char_ratio, alpha_char_ratio |

---

##### Task 2: 문자 구성 특성
<small>**Sherlock 기반 15개 피처 | 문자 비율 분석 **</small>

| 범주 | 피처 |
|------|------|
| 문자 비율 | digit_ratio, alpha_ratio, space_ratio, upper_ratio, special_ratio |
| 문자 비율 std | digit_ratio_std, alpha_ratio_std, upper_ratio_std |
| 문자열 길이 | avg_str_len, str_len_std |
| 단어 길이 | word_len_mean, word_len_std, word_len_min, word_len_max, distinct_char_ratio |

---

##### Task 4: 통합 특성
<small>**Task 1, 2 특징 통합 | 42 차원**</small>


In [ ]:
# [v13 cell 2] FeatureExtractor — 18 stat + 18 char ratio features
import os
import re
import warnings
import glob
import numpy as np
import pandas as pd
from scipy import stats as scstats
from collections import Counter
from tqdm.auto import tqdm

# Regex helpers (compile once)
_RE_ID_LIKE   = re.compile(r"^[A-Za-z]+[_\-]?\d+$|^\d+$")
_RE_CODE_LIKE = re.compile(r"^[A-Z0-9]{2,8}$")
_RE_NAME_LIKE = re.compile(r"^[A-Z][a-zA-Z\s\-\.]+$")
_RE_YEAR_LIKE = re.compile(r"^(19|20)\d{2}$")
_RE_DATE_LIKE = re.compile(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}|^\d{1,2}[-/]\d{1,2}[-/]\d{2,4}")
_RE_EMAIL     = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
_RE_URL       = re.compile(r"^https?://|^www\.")
_DELIMS       = set("-_/.")


class FeatureExtractor:
    """v13 spec: 18 stat ratio features + 18 char shape ratio features = 36-dim per column"""

    STAT_DIM = 18
    CHAR_DIM = 18
    TOTAL_DIM = STAT_DIM + CHAR_DIM   # 36

    def __init__(self, dataset_path):
        self.dataset_path = dataset_path

    # ── 18 stat ratio features ──────────────────────────────────────
    def _stat_features(self, series):
        s = series.dropna()
        n_raw = len(series)
        n = len(s)
        if n == 0:
            return np.zeros(self.STAT_DIM, dtype=np.float32)

        n_unique = s.nunique()
        unique_ratio = n_unique / n
        dup_ratio = 1.0 - unique_ratio

        vc = s.value_counts()
        avg_freq = float(vc.mean())
        median_freq = float(vc.median())
        top1_ratio = float(vc.iloc[0] / n) if len(vc) > 0 else 0.0
        top5_ratio = float(vc.head(5).sum() / n) if len(vc) > 0 else 0.0

        # entropy (normalized by log(n_unique))
        probs = vc.values / n
        entropy = float(-(probs * np.log(probs + 1e-12)).sum()) / max(np.log(max(n_unique, 2)), 1e-6)

        # numeric-castable ratio
        nums = pd.to_numeric(s, errors="coerce")
        valid = nums.dropna()
        numeric_parse_ratio = len(valid) / n
        integer_ratio = (valid % 1 == 0).mean() if len(valid) > 0 else 0.0
        decimal_ratio = (valid % 1 != 0).mean() if len(valid) > 0 else 0.0

        if len(valid) > 1:
            v_arr = valid.astype(float).values
            v_min = float(v_arr.min())
            v_max = float(v_arr.max())
            v_mean = float(v_arr.mean())
            v_med = float(np.median(v_arr))
            v_std = float(v_arr.std(ddof=0))
            min_max_ratio = (v_min / v_max) if v_max != 0 else 0.0
            range_per_median = (v_max - v_min) / (abs(v_med) + 1e-6)
            std_per_mean = v_std / (abs(v_mean) + 1e-6)
            # v13 안정성: variance 너무 작으면 skew/kurt 의미 없음 → 0으로 처리
            if v_std < 1e-9:
                skewness, kurtosis = 0.0, 0.0
            else:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    try:
                        skewness = float(scstats.skew(v_arr, bias=False))
                        kurtosis = float(scstats.kurtosis(v_arr, bias=False))
                        if np.isnan(skewness) or np.isinf(skewness):
                            skewness = 0.0
                        if np.isnan(kurtosis) or np.isinf(kurtosis):
                            kurtosis = 0.0
                    except Exception:
                        skewness, kurtosis = 0.0, 0.0
        else:
            min_max_ratio = range_per_median = std_per_mean = skewness = kurtosis = 0.0

        null_ratio = (n_raw - n) / max(n_raw, 1)

        feats = [
            np.log1p(n),               # 0 n_rows (log)
            null_ratio,                # 1
            unique_ratio,              # 2
            dup_ratio,                 # 3
            avg_freq / n,              # 4 normalized avg_freq
            median_freq / n,           # 5 normalized median_freq
            min_max_ratio,             # 6
            np.tanh(range_per_median), # 7 squashed
            np.tanh(std_per_mean),     # 8 squashed
            np.tanh(skewness),         # 9 squashed
            np.tanh(kurtosis / 10),    # 10 squashed
            entropy,                   # 11
            top1_ratio,                # 12
            top5_ratio,                # 13
            numeric_parse_ratio,       # 14
            integer_ratio,             # 15
            decimal_ratio,             # 16
            np.log1p(n_unique) / max(np.log1p(n), 1e-6),   # 17 log_distinct_ratio
        ]
        return np.array(feats, dtype=np.float32)

    # ── 18 char shape ratio features (v30 옵션C — 8 count는 단일 numpy 패스/풀데이터, token+regex 10개는 50K 샘플) ──
    def _char_features(self, series):
        s = series.dropna().astype(str)
        if len(s) == 0:
            return np.zeros(self.CHAR_DIM, dtype=np.float32)

        # ── 8 char-class ratios: 단일 numpy 패스 (ASCII, 풀 데이터) ──
        # 모든 문자열 합쳐서 uint8 배열로 → 한 번에 분류 (모델 영향 0)
        joined = "".join(s.values)
        if len(joined) == 0:
            return np.zeros(self.CHAR_DIM, dtype=np.float32)
        arr = np.frombuffer(joined.encode("utf-8", errors="ignore"), dtype=np.uint8)
        total_chars = arr.size
        if total_chars == 0:
            return np.zeros(self.CHAR_DIM, dtype=np.float32)

        is_upper = (arr >= 65) & (arr <= 90)     # A-Z
        is_lower = (arr >= 97) & (arr <= 122)    # a-z
        is_alpha = is_upper | is_lower
        is_digit = (arr >= 48) & (arr <= 57)     # 0-9
        # 공백: space, tab, newline, CR
        is_space = (arr == 32) | (arr == 9) | (arr == 10) | (arr == 13)
        # delim: - _ / .   (codes: 45, 95, 47, 46)
        is_delim = (arr == 45) | (arr == 95) | (arr == 47) | (arr == 46)
        is_alnum = is_alpha | is_digit
        is_special = ~(is_alnum | is_space | is_delim)

        alpha = int(is_alpha.sum()); digit = int(is_digit.sum())
        upper = int(is_upper.sum()); lower = int(is_lower.sum())
        space = int(is_space.sum()); delim = int(is_delim.sum())
        alnum = alpha + digit
        special = int(is_special.sum())

        alpha_ratio   = alpha   / total_chars
        digit_ratio   = digit   / total_chars
        alnum_ratio   = alnum   / total_chars
        upper_ratio   = upper   / total_chars
        lower_ratio   = lower   / total_chars
        space_ratio   = space   / total_chars
        delim_ratio   = delim   / total_chars
        special_ratio = special / total_chars

        # ── token 통계 + regex 매치: 50K 샘플 (모델 영향 ~0.5%) ──
        TOKREGEX_CAP = 50000
        if len(s) > TOKREGEX_CAP:
            s_sample = s.sample(TOKREGEX_CAP, random_state=0)
        else:
            s_sample = s

        tokens = s_sample.str.strip().str.split(r"[\s_\-/.]+", regex=True)
        token_counts = tokens.str.len()
        avg_token_count = float(token_counts.mean()) if len(token_counts) > 0 else 0.0

        flat = tokens.explode().dropna()
        flat = flat[flat.str.len() > 0]
        if len(flat) > 0:
            flat_lens = flat.str.len()
            avg_token_len = float(flat_lens.mean())
            max_token_len = float(flat_lens.max())
        else:
            avg_token_len = max_token_len = 0.0

        n_rows = len(s_sample)
        id_like    = float(s_sample.str.match(_RE_ID_LIKE).sum())    / n_rows
        code_like  = float(s_sample.str.match(_RE_CODE_LIKE).sum())  / n_rows
        name_like  = float(s_sample.str.match(_RE_NAME_LIKE).sum())  / n_rows
        year_like  = float(s_sample.str.match(_RE_YEAR_LIKE).sum())  / n_rows
        date_like  = float(s_sample.str.match(_RE_DATE_LIKE).sum())  / n_rows
        email_like = float(s_sample.str.match(_RE_EMAIL).sum())      / n_rows
        url_like   = float(s_sample.str.match(_RE_URL).sum())        / n_rows

        feats = [
            alpha_ratio, digit_ratio, alnum_ratio,
            upper_ratio, lower_ratio, space_ratio,
            delim_ratio, special_ratio,
            np.tanh(avg_token_count / 5),
            np.tanh(avg_token_len / 10),
            np.tanh(max_token_len / 20),
            id_like, code_like, name_like,
            year_like, date_like, email_like, url_like,
        ]
        return np.array(feats, dtype=np.float32)

    # ── public ────────────────────────────────────────────────────
    def load_tpcds_data(self):
        """Return dict of {filename: dataframe} for every CSV in dataset_path.
        Used by Cell 12 to build Xdata cache (all training pool columns)."""
        all_data = {}
        for fp in tqdm(sorted(glob.glob(os.path.join(self.dataset_path, "*.csv"))), desc="FE pre-scan"):
            try:
                df = pd.read_csv(fp, low_memory=False)
                all_data[os.path.basename(fp)] = df
            except Exception:
                continue
        return all_data

    def task4_combine_features(self, series):
        """Single column → 36-dim (18 stat + 18 char)"""
        stat = self._stat_features(series)
        char = self._char_features(series)
        return np.concatenate([stat, char])


# Auto-instantiate extractor for later cells (mimics existing v7 pattern)
if 'extractor' not in globals():
    extractor = None  # will be created later when dataset_path is known


#### 3. Overlap 계산

##### Overlap 계산 + 전체 통합
<small>**4개 피처 | Jaccard + Containment A→B + Containment B→A + Has_overlap**</small>

| 작업 | 설명 |
|------|------|
| 계산 대상 | raw_dataset의 모든 고유 쌍 |
| 함수 | `compute_overlap_features()` |
| 출력 | `[jaccard, con_A→B, con_B→A, has_overlap]` |


In [ ]:
# [v14 cell 3] compute_overlap_features — 14-dim
import json
import random
import pandas as pd
import numpy as np
import os
import gc

csv_cache = {}

def clear_memory():
    if 'csv_cache' in globals():
        csv_cache.clear()
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def compute_overlap_features(series_A, series_B):
    vals_A_full = series_A.dropna().astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    vals_B_full = series_B.dropna().astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

    vals_A = set(vals_A_full)
    vals_B = set(vals_B_full)

    nA_raw = len(vals_A_full)
    nB_raw = len(vals_B_full)

    if nA_raw == 0 or nB_raw == 0 or not vals_A or not vals_B:
        return [0.0] * 20

    nuniq_A = len(vals_A)
    nuniq_B = len(vals_B)

    unique_ratio_A = nuniq_A / nA_raw
    unique_ratio_B = nuniq_B / nB_raw
    max_unique_ratio = max(unique_ratio_A, unique_ratio_B)
    min_unique_ratio = min(unique_ratio_A, unique_ratio_B)

    intersection = vals_A & vals_B
    union = vals_A | vals_B

    inter_size = len(intersection)
    union_size = len(union)

    jaccard = inter_size / union_size if union_size > 0 else 0.0
    con_a = inter_size / nuniq_A if nuniq_A > 0 else 0.0
    con_b = inter_size / nuniq_B if nuniq_B > 0 else 0.0

    containment_max = max(con_a, con_b)
    containment_min = min(con_a, con_b)
    containment_gap = abs(con_a - con_b)

    fk_direction_score = con_b - con_a
    one_sided_full = 1.0 if (con_a >= 0.999 or con_b >= 0.999) else 0.0

    intersection_size_ratio = inter_size / max(min(nuniq_A, nuniq_B), 1)
    union_size_ratio = union_size / max(nuniq_A + nuniq_B, 1)

    both_low_unique_flag = 1.0 if (max_unique_ratio < 0.5) else 0.0
    low_cardinality_flag = 1.0 if (min(nuniq_A, nuniq_B) <= 3) else 0.0
    binary_like_flag = 1.0 if (min(nuniq_A, nuniq_B) <= 2) else 0.0

    one_to_many_likelihood = unique_ratio_A * (1.0 - unique_ratio_B) * con_b
    many_to_one_likelihood = (1.0 - unique_ratio_A) * unique_ratio_B * con_a

    # v21 신규: cardinality/frequency 비대칭 신호 (signed)
    import math
    log_card_ratio = math.log(nuniq_A + 1) - math.log(nuniq_B + 1)
    freq_A = nA_raw / max(nuniq_A, 1)
    freq_B = nB_raw / max(nuniq_B, 1)
    log_freq_ratio = math.log(freq_A + 1) - math.log(freq_B + 1)

    # v30 신규: 분포 유사도 신호 4개 (양쪽 분포가 비슷한 trap vs 진짜 FK 구분)
    vc_A = vals_A_full.value_counts()
    vc_B = vals_B_full.value_counts()

    # 16: freq_dist_diff — top-10 빈도 분포 L1 차이 (양쪽 카테고리 trap은 매우 낮음)
    top10_A = list(vc_A.head(10).values / max(nA_raw, 1)) if len(vc_A) > 0 else []
    top10_B = list(vc_B.head(10).values / max(nB_raw, 1)) if len(vc_B) > 0 else []
    while len(top10_A) < 10: top10_A.append(0.0)
    while len(top10_B) < 10: top10_B.append(0.0)
    freq_dist_diff = float(sum(abs(top10_A[i] - top10_B[i]) for i in range(10)))

    # 17: bigram_entropy_diff — 양쪽 char bigram entropy 차이 (symmetric)
    def _bigram_entropy(values, max_sample=500):
        from collections import Counter as _Counter
        if not values: return 0.0
        sample = values[:max_sample] if len(values) > max_sample else values
        bigrams = []
        for v in sample:
            for i in range(len(v) - 1):
                bigrams.append(v[i:i+2])
        if not bigrams: return 0.0
        cnt = _Counter(bigrams)
        total = sum(cnt.values())
        probs = [c / total for c in cnt.values()]
        H = -sum(p * math.log(p + 1e-9) for p in probs)
        return H / math.log(len(probs) + 1)   # normalized

    _values_A = list(vals_A_full)
    _values_B = list(vals_B_full)
    ent_A = _bigram_entropy(_values_A)
    ent_B = _bigram_entropy(_values_B)
    bigram_entropy_diff = abs(ent_A - ent_B)

    # 18: top1_concentration_diff — 양쪽 top1 비율 차이 (양쪽 카테고리 trap은 매우 낮음)
    top1_A = float(vc_A.iloc[0] / max(nA_raw, 1)) if len(vc_A) > 0 else 0.0
    top1_B = float(vc_B.iloc[0] / max(nB_raw, 1)) if len(vc_B) > 0 else 0.0
    top1_concentration_diff = abs(top1_A - top1_B)

    # 19: cardinality_class_match — 양쪽 nuniq class 일치 여부 (binary/small/medium/large)
    def _card_class(nu):
        if nu <= 2:   return 0
        if nu <= 20:  return 1
        if nu <= 200: return 2
        return 3
    cardinality_class_match = 1.0 if _card_class(nuniq_A) == _card_class(nuniq_B) else 0.0

    return [
        float(jaccard),
        float(con_a),
        float(con_b),
        float(containment_max),
        float(containment_min),
        float(containment_gap),
        float(fk_direction_score),
        float(one_sided_full),
        float(intersection_size_ratio),
        float(union_size_ratio),
        float(max_unique_ratio),
        float(min_unique_ratio),
        float(both_low_unique_flag),
        float(low_cardinality_flag),
        float(log_card_ratio),               # 14: signed
        float(log_freq_ratio),               # 15: signed
        # v30 신규 4개 (분포 유사도, 모두 symmetric)
        float(freq_dist_diff),               # 16
        float(bigram_entropy_diff),          # 17
        float(top1_concentration_diff),      # 18
        float(cardinality_class_match),      # 19
    ]


#### 4. 모델 아키텍처

<small>**combined_dim = single_input_dim × 2 + extra_pair_features = 84 + 4(overlap_feature)**</small>

---
<pre style="padding: 10px; border-radius: 5px;">
Input (172차원)
    │
    ├─ diff (84:126, 42차원)
    ├─ mul (126:168, 42차원)
    └─ ov (168:172, 4차원)
         │
         ├─────────────────────────────┬─────────────────┐
         │                             │                 │
    
    f_stat (54)                f_char (30)              ov (6)
    diff[0:27]                 diff[27:42]
    + mul[0:27]                + mul[27:42]
         │                           │                   │
         │                           │                   │
    branch_stat                branch_char         branch_overlap
    54→32 (ReLU)             30→32 (ReLU)         6→64 (ReLU)
         │                           │                   │
         └───────────────┬───────────┴───────────────────┘
                         │
                   concat (128)
                         │
                  Linear(128→64)
                         │
                        ReLU
                         │
                  Linear(64→2)
                         │
            ┌────────────┴────────────┐
            │                         │
        비매칭(0)                    매칭(1)
</pre>



In [ ]:
# [v33 ablation] V33AblationMatcher — branch_config로 7가지 변형 지원
# v33 학습 데이터 (trap1~5 포함) + V14_Matcher 구조 (subtract fusion)
import torch
import torch.nn as nn
import torch.nn.functional as F


class V33AblationMatcher(nn.Module):
    """v33 Ablation: branch_config = subset of ['stat', 'char', 'ov']
    예: ['stat'], ['char'], ['ov'], ['stat','char'], ['stat','ov'], ['char','ov'], ['stat','char','ov']
    입력 98-dim. fusion = subtract style ([a, b, |a-b|]) — v33 baseline 동일.
    Layout: u_stat(0:18) u_char(18:36) v_stat(36:54) v_char(54:72) pair_feat(72:98)
            pair_feat = 6 pair_stat + 20 overlap (v30 신규 4-dim 포함)
    """

    EXPECTED_INPUT = 98
    D_STAT = 18
    D_CHAR = 18
    D_PAIR_STAT = 6
    D_OVERLAP = 20

    def __init__(self, branch_config, e_stat=32, e_char=32, e_pair=32, dropout=0.15):
        super().__init__()
        self.branch_config = tuple(sorted(branch_config))
        self.use_stat = 'stat' in branch_config
        self.use_char = 'char' in branch_config
        self.use_ov = 'ov' in branch_config
        assert self.use_stat or self.use_char or self.use_ov, "At least one branch required"

        if self.use_stat:
            self.stat_enc = nn.Sequential(
                nn.Linear(self.D_STAT, 64), nn.BatchNorm1d(64), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(64, e_stat), nn.ReLU(),
            )
        if self.use_char:
            self.char_enc = nn.Sequential(
                nn.Linear(self.D_CHAR, 64), nn.BatchNorm1d(64), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(64, e_char), nn.ReLU(),
            )
        if self.use_ov:
            self.pair_enc = nn.Sequential(
                nn.Linear(self.D_PAIR_STAT + self.D_OVERLAP, 64),  # 6+20=26
                nn.BatchNorm1d(64), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(64, e_pair), nn.ReLU(),
            )

        fusion_dim = 0
        if self.use_stat: fusion_dim += e_stat * 3   # a_s, b_s, |a_s-b_s|
        if self.use_char: fusion_dim += e_char * 3
        if self.use_ov:   fusion_dim += e_pair

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, 1),
        )

    def forward(self, x):
        u_stat = x[:, 0:18]
        u_char = x[:, 18:36]
        v_stat = x[:, 36:54]
        v_char = x[:, 54:72]
        pair_feat = x[:, 72:98]   # 6 pair_stat + 20 overlap = 26
        parts = []
        if self.use_stat:
            a_s = self.stat_enc(u_stat); b_s = self.stat_enc(v_stat)
            parts.extend([a_s, b_s, (a_s - b_s).abs()])
        if self.use_char:
            a_c = self.char_enc(u_char); b_c = self.char_enc(v_char)
            parts.extend([a_c, b_c, (a_c - b_c).abs()])
        if self.use_ov:
            parts.append(self.pair_enc(pair_feat))
        fused = torch.cat(parts, dim=1)
        return self.classifier(fused).squeeze(-1)


# Backward-compat (기본 풀 3 브랜치 = v33 baseline)
def CRIM_Statisticsnly_Matcher():
    return V33AblationMatcher(branch_config=['stat', 'char', 'ov'])

V14_Matcher = lambda: V33AblationMatcher(branch_config=['stat', 'char', 'ov'])


#### 5. 데이터 수집 및 특징 추출

**처리 단계**
1. CSV 파일에서 데이터 로드
2. 각 컬럼마다 **1200개 특징** 추출
3. **vstack**: list → 2차원 행렬로 변환

**캐시 저장**
- `features_cache_path`: 추출된 특징 저장
- `metadata_cache_path`: 파일 & 컬럼 메타데이터 저장
  - 형식: `{'file': 'orders.csv', 'column': 'order_id'}`

**네거티브 샘플 생성**
- metadata_cache를 활용하여 양성 쌍(FK) 정보 확인
- **양성 쌍이 아닐 때만** label=0 (비관계)으로 학습 데이터 추가
- 결과: 양성 쌍(label=1) + 네거티브 쌍(label=0)의 균형잡힌 학습 데이터셋


In [ ]:
import os
import numpy as np
import pickle
import pandas as pd
from tqdm.auto import tqdm

cache_dir = os.path.join(base_path, 'cache')
os.makedirs(cache_dir, exist_ok=True)

def is_spider_file_name(name):
    base = str(name).lower()
    return base.startswith('spd_') or 'spider' in base

def has_excluded_source(metadata):
    return any(is_spider_file_name(str(item.get('file', ''))) for item in metadata)

cache_is_valid = os.path.exists(features_cache_path) and os.path.exists(metadata_cache_path)
if cache_is_valid:
    with open(metadata_cache_path, 'rb') as f:
        column_metadata = pickle.load(f)
    if has_excluded_source(column_metadata):
        print('stale cache contains spider sources -> rebuilding TPC-only cache')
        for stale_cache_path in [features_cache_path, metadata_cache_path]:
            if os.path.exists(stale_cache_path):
                os.remove(stale_cache_path)
        cache_is_valid = False
        column_metadata = []

if cache_is_valid:
    print('📦 Cache Found...')
    Xdata = np.load(features_cache_path, allow_pickle=True)
    print(f'Loaded {Xdata.shape[0]} cached feature vectors')
else:
    print('🚀 First Run: extracting TPC-only features...')
    extractor = FeatureExtractor(dataset_path)
    all_data = extractor.load_tpcds_data()
    tpc_data = {k: v for k, v in all_data.items() if not is_spider_file_name(k)}

    all_features = []
    column_metadata = []
    for filename, df in tpc_data.items():
        print(f'Processing {filename}...')
        for colname in tqdm(df.columns, desc=f'Extracting {filename}'):
            out = extractor.task4_combine_features(df[colname])
            feat_vector = np.array(out).flatten()
            all_features.append(feat_vector)
            column_metadata.append({'file': filename, 'column': colname})

    Xdata = np.vstack(all_features)
    np.save(features_cache_path, Xdata)
    with open(metadata_cache_path, 'wb') as f:
        pickle.dump(column_metadata, f)
    print(f'✅ Saved TPC-only feature cache to {cache_dir}')

print('-' * 30)
print(f'Xdata Shape: {Xdata.shape}')
print(f'First 5 sample values: {Xdata[0][:5]}')


6. 학습용 라벨링 및 Dataset 생성
<pre>
[ground_truth]
    │
    ├─────────────────────────────────────┐
    │                                     │
    
Positive 샘플 생성                Negative 샘플 생성
85% 샘플링                       positive 샘플 수만큼 추출
5번 × 2(양방향)                  ground_truth에 없는 쌍
label = 1                         샘플링 없음
                                  label = 0
    │                                     │
    └─────────────────┬───────────────────┘
                      │
                 raw_dataset
              (Positive + Negative)
                      │
         ┌────────────┼────────────┐
         │            │            │
    컬럼1특징        컬럼2특징        OV특징
         │            │            │
         └────────────┼────────────┘
                      │
                  라벨(0,1)
                      │
                   컬럼쌍
                      │
                final_dataset
</pre>


In [ ]:
# [v31 cell 14] Data construction — definitions only (실행은 sub-cells에서)
import json
import random
import pandas as pd
import numpy as np
import os
import pickle
import gc
from tqdm.auto import tqdm

# ── 체크포인트 디렉토리 ─────────────────────────────────────────
CKPT_DIR = os.path.join(base_path, f'ckpt_{post_fix}')
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'📁 Checkpoint dir: {CKPT_DIR}')

def ckpt_path(name):
    return os.path.join(CKPT_DIR, f'{name}_{post_fix}.pkl')

def load_ckpt(name):
    p = ckpt_path(name)
    if os.path.exists(p):
        with open(p, 'rb') as f:
            return pickle.load(f)
    return None

def save_ckpt(name, obj):
    p = ckpt_path(name)
    with open(p, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  ✅ saved: {p}')

# ── 1) Data source 라우팅 ─────────────────────────────────────────
SOURCE_PATHS = [TPC_H_PATH, TPC_DS_PATH, SPIDER_PATH]

def get_df(table_name):
    clean_name = table_name.replace('.csv', '')
    cache_key = clean_name
    if cache_key in csv_cache:
        return csv_cache[cache_key]
    for src_path in SOURCE_PATHS:
        fp = os.path.join(src_path, f'{clean_name}.csv')
        if os.path.exists(fp):
            df = pd.read_csv(fp, low_memory=False)
            csv_cache[cache_key] = df
            return df
    raise FileNotFoundError(f'CSV not found for table: {clean_name}')


def get_feat_for_column(df, colname):
    if colname not in df.columns:
        raise ValueError(f'{colname} not found')
    out = extractor.task4_combine_features(df[colname])
    return np.array(out).flatten()


def is_spider_table_name(name):
    return str(name).lower().startswith('spd_')


if extractor is None:
    extractor = FeatureExtractor(dataset_path)


# ── 2) Ground truth 로드 + 데이터 소스별 FK 분리 ──────────────────
FORBIDDEN_DBS = {'toxicology', 'california_schools', 'superhero', 'thrombosis_prediction'}

with open(ground_truth_path, 'r', encoding='utf-8') as f:
    gt_data = json.load(f)

tpcds_fk = [(e['child'][0], e['child'][1], e['parent'][0], e['parent'][1])
            for e in gt_data.get('tpcds', {}).get('fk_pairs', [])]
tpch_fk = [(e['child'][0], e['child'][1], e['parent'][0], e['parent'][1])
           for e in gt_data.get('sf1', {}).get('fk_pairs', [])]
spider_fk = [(e['child'][0], e['child'][1], e['parent'][0], e['parent'][1])
             for db_name, src_info in gt_data.items()
             if db_name not in ('sf1', 'tpcds', 'travel_log') and db_name not in FORBIDDEN_DBS
             for e in src_info.get('fk_pairs', [])]

print(f'TPC-DS FK: {len(tpcds_fk)}, TPC-H FK: {len(tpch_fk)}, SPIDER FK: {len(spider_fk)}')

# pos_set: 양방향 FK 집합 (모든 sub-cell에서 사용)
pos_set = set()
for fk_list in [tpcds_fk, tpch_fk, spider_fk]:
    for t1, c1, t2, c2 in fk_list:
        pos_set.add((t1, c1, t2, c2))
        pos_set.add((t2, c2, t1, c1))


# ── 3) augment 파라미터 + 헬퍼 ────────────────────────────────────
RATIO_TPC_DS = 6
RATIO_TPC_H  = 3
RATIO_SPIDER = 1
TARGET_TOTAL_POS = 1000
target_tpcds = int(TARGET_TOTAL_POS * RATIO_TPC_DS / (RATIO_TPC_DS + RATIO_TPC_H + RATIO_SPIDER))
target_tpch  = int(TARGET_TOTAL_POS * RATIO_TPC_H  / (RATIO_TPC_DS + RATIO_TPC_H + RATIO_SPIDER))
target_spider= int(TARGET_TOTAL_POS * RATIO_SPIDER / (RATIO_TPC_DS + RATIO_TPC_H + RATIO_SPIDER))
SAMPLING_FRAC = 0.85


def add_pair(raw_dataset, existing_keys, f1, f2, label, key):
    if key in existing_keys:
        return False
    raw_dataset.append((f1, f2, label, key))
    existing_keys.add(key)
    return True


def augment_fks(fk_list, target_pos_count, tag, raw_dataset, existing_keys):
    if not fk_list:
        return 0
    aug_per_fk = max(1, target_pos_count // len(fk_list))
    print(f'[{tag}] {len(fk_list)} FK × {aug_per_fk} aug → target {target_pos_count} (단방향)')
    added = 0
    for fk_idx, (t1, c1, t2, c2) in enumerate(tqdm(fk_list, desc=tag)):
        try:
            df1 = get_df(t1); df2 = get_df(t2)
            for k in range(aug_per_fk):
                sub1 = df1.sample(frac=SAMPLING_FRAC, random_state=42 + k).reset_index(drop=True)
                sub2 = df2.sample(frac=SAMPLING_FRAC, random_state=142 + k).reset_index(drop=True)
                f1 = get_feat_for_column(sub1, c1)
                f2 = get_feat_for_column(sub2, c2)
                add_pair(raw_dataset, existing_keys, f1, f2, 1, (t1, c1, t2, c2, tag, k, 'fwd'))
                add_pair(raw_dataset, existing_keys, f2, f1, 1, (t2, c2, t1, c1, tag, k, 'rev'))
                added += 2
                del sub1, sub2, f1, f2
                if added >= target_pos_count * 2:
                    break
        except Exception:
            continue
        gc.collect()
        if (fk_idx + 1) % 10 == 0:
            csv_cache.clear()
            gc.collect()
            try:
                import torch as _torch
                _torch.cuda.empty_cache()
            except Exception:
                pass
        if added >= target_pos_count * 2:
            break
    csv_cache.clear()
    gc.collect()
    return added


# ── 4) v30 overlap reverse helper (모든 후속 cell에서 사용) ──────
def _reverse_overlap_v30(ov):
    """v30 reverse 20-dim — con_a/con_b swap, fk_direction/log_card/log_freq negate, 나머지 sym."""
    return [
        ov[0], ov[2], ov[1], ov[3], ov[4], ov[5],
        -ov[6], ov[7], ov[8], ov[9],
        ov[10], ov[11], ov[12], ov[13],
        -ov[14], -ov[15],
        ov[16], ov[17], ov[18], ov[19],
    ]


# SPIDER 가용성
spider_available = os.path.exists(SPIDER_PATH) and len(os.listdir(SPIDER_PATH)) > 0
if not spider_available:
    print('⚠ SPIDER_PATH 비어있음 → SPIDER 양성 건너뜀')

print('\n✅ Cell 14 정의 완료. 다음 sub-cell들 순서대로 실행:')
print('  14a → 15a → 15b → 15c → 15d → 15e → 15f')


In [ ]:
# [v31 cell 14a] 양성 FK augmentation 실행 + 체크포인트
# 체크포인트가 있으면 로드, 없으면 augment 실행

_ckpt_name = 'raw_dataset_positives'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    raw_dataset = _loaded['raw_dataset']
    existing_keys = _loaded['existing_keys']
    pos_count = _loaded['pos_count']
    print(f'  raw_dataset size: {len(raw_dataset)}, pos_count: {pos_count}')
else:
    print('🚀 Computing positives from scratch...')
    raw_dataset = []
    existing_keys = set()

    augment_fks(tpcds_fk, target_tpcds, 'TPC-DS', raw_dataset, existing_keys)
    augment_fks(tpch_fk,  target_tpch,  'TPC-H',  raw_dataset, existing_keys)
    if spider_available:
        random.seed(42)
        n_spider_fks = max(1, target_spider // 2)
        spider_sample = random.sample(spider_fk, min(n_spider_fks, len(spider_fk)))
        augment_fks(spider_sample, target_spider, 'SPIDER', raw_dataset, existing_keys)

    pos_count = sum(1 for _, _, y, _ in raw_dataset if y == 1)
    print(f'\nTotal positives: {pos_count}')

    save_ckpt(_ckpt_name, {
        'raw_dataset': raw_dataset,
        'existing_keys': existing_keys,
        'pos_count': pos_count,
    })

clear_memory()


In [ ]:
# [v31 cell 15a] Random negatives + 체크포인트
_ckpt_name = 'dataset_with_randneg'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    raw_dataset = _loaded['raw_dataset']
    existing_keys = _loaded['existing_keys']
    pos_count = _loaded['pos_count']
    target_neg = _loaded['target_neg']
    random_added = _loaded['random_added']
    print(f'  raw_dataset size: {len(raw_dataset)}, random_added: {random_added}')
else:
    print('🚀 Adding random negatives...')

    if 'raw_dataset' not in locals() or 'pos_count' not in locals():
        prev = load_ckpt('raw_dataset_positives')
        assert prev is not None, '14a 체크포인트 없음 — 먼저 Cell 14a 실행'
        raw_dataset = prev['raw_dataset']
        existing_keys = prev['existing_keys']
        pos_count = prev['pos_count']

    target_neg = pos_count * 2

    if 'Xdata' not in locals() or 'column_metadata' not in locals():
        raise FileNotFoundError('Feature cache (Xdata, column_metadata) missing — run Cell 12 first')

    target_random_neg = int(target_neg * 0.4)
    print(f'[NEG] target random = {target_random_neg}, target hard = {target_neg - target_random_neg}')

    random_added = 0
    attempts = 0
    while random_added < target_random_neg and attempts < target_random_neg * 50:
        idx1, idx2 = random.sample(range(len(Xdata)), 2)
        m1, m2 = column_metadata[idx1], column_metadata[idx2]
        t1n, c1n = m1['file'].replace('.csv', ''), m1['column']
        t2n, c2n = m2['file'].replace('.csv', ''), m2['column']
        pairkey = (t1n, c1n, t2n, c2n, 'rand_neg')
        base = (t1n, c1n, t2n, c2n)
        if base not in pos_set and pairkey not in existing_keys:
            raw_dataset.append((Xdata[idx1], Xdata[idx2], 0, pairkey))
            existing_keys.add(pairkey)
            random_added += 1
        attempts += 1
    print(f'random neg added: {random_added}')

    save_ckpt(_ckpt_name, {
        'raw_dataset': raw_dataset,
        'existing_keys': existing_keys,
        'pos_count': pos_count,
        'target_neg': target_neg,
        'random_added': random_added,
    })

clear_memory()


In [ ]:
# [v31 cell 15b] Overlap features 계산 (가장 무거운 단계, ~25분) + 체크포인트
_ckpt_name = 'pair_overlap'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    pair_overlap = _loaded
    print(f'  pair_overlap size: {len(pair_overlap)}')
else:
    print('🚀 Computing overlap for unique pairs (이 단계가 가장 오래 걸림)...')

    if 'raw_dataset' not in locals():
        prev = load_ckpt('dataset_with_randneg')
        assert prev is not None, '15a 체크포인트 없음'
        raw_dataset = prev['raw_dataset']
        existing_keys = prev['existing_keys']

    pair_overlap = {}
    unique_pairs = []
    for _, _, _, pairkey in raw_dataset:
        if len(pairkey) >= 4:
            base = (pairkey[0], pairkey[1], pairkey[2], pairkey[3])
            unique_pairs.append(base)
    unique_pairs = list(dict.fromkeys(unique_pairs))
    print(f'[OV] computing overlap for {len(unique_pairs)} unique pairs...')

    for _ov_idx, (t1, c1, t2, c2) in enumerate(tqdm(unique_pairs, desc='overlap')):
        if (t1, c1, t2, c2) in pair_overlap:
            continue
        try:
            ov = compute_overlap_features(get_df(t1)[c1], get_df(t2)[c2])
            pair_overlap[(t1, c1, t2, c2)] = ov
            pair_overlap[(t2, c2, t1, c1)] = _reverse_overlap_v30(ov)
        except Exception:
            pair_overlap[(t1, c1, t2, c2)] = [0.0] * 20
            pair_overlap[(t2, c2, t1, c1)] = [0.0] * 20
        if (_ov_idx + 1) % 100 == 0:
            csv_cache.clear()
            gc.collect()

    save_ckpt(_ckpt_name, pair_overlap)

clear_memory()


In [ ]:
# [v31 cell 15c] Hard negatives (2종 gate + final_dataset 생성) + 체크포인트
_ckpt_name = 'final_dataset_step1'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    final_dataset = _loaded['final_dataset']
    existing_keys = _loaded['existing_keys']
    pair_overlap  = _loaded['pair_overlap']
    hard_added    = _loaded['hard_added']
    print(f'  final_dataset size: {len(final_dataset)}, hard_added: {hard_added}')
else:
    print('🚀 Building hard negatives...')

    if 'pair_overlap' not in locals():
        pair_overlap = load_ckpt('pair_overlap')
        assert pair_overlap is not None, '15b 체크포인트 없음'
    if 'raw_dataset' not in locals():
        prev = load_ckpt('dataset_with_randneg')
        raw_dataset = prev['raw_dataset']
        existing_keys = prev['existing_keys']
        target_neg = prev['target_neg']

    def is_high_overlap_hard_negative(ov):
        (jaccard, con_a, con_b, containment_max, containment_min, containment_gap,
         fk_direction_score, one_sided_full, intersection_size_ratio, union_size_ratio,
         max_unique_ratio, min_unique_ratio, both_low_unique_flag, low_cardinality_flag,
         log_card_ratio, log_freq_ratio,
         freq_dist_diff, bigram_entropy_diff, top1_concentration_diff, cardinality_class_match) = ov
        return (containment_max >= 0.85 and jaccard >= 0.30 and
                containment_gap <= 0.20 and max_unique_ratio <= 0.60 and
                (both_low_unique_flag == 1.0 or low_cardinality_flag == 1.0))

    def is_low_cardinality_trap_negative(ov, nuniq_A, nuniq_B):
        (jaccard, con_a, con_b, containment_max, containment_min, containment_gap,
         fk_direction_score, one_sided_full, intersection_size_ratio, union_size_ratio,
         max_unique_ratio, min_unique_ratio, both_low_unique_flag, low_cardinality_flag,
         log_card_ratio, log_freq_ratio,
         freq_dist_diff, bigram_entropy_diff, top1_concentration_diff, cardinality_class_match) = ov
        min_card = min(nuniq_A, nuniq_B); max_card = max(nuniq_A, nuniq_B)
        return (min_card <= 3 and containment_max >= 0.95 and jaccard >= 0.80 and max_card <= 10)

    hard_candidates = []
    # v31 patch: nuniq 캐시로 hard cand scan 가속 (이전 12분 → 1분)
    nuniq_cache = {}
    def _get_nuniq(t, c):
        key = (t, c)
        if key in nuniq_cache:
            return nuniq_cache[key]
        try:
            v = get_df(t)[c].dropna().astype(str).nunique()
        except Exception:
            v = 0
        nuniq_cache[key] = v
        return v

    for (t1, c1, t2, c2), ov in tqdm(pair_overlap.items(), desc='hard-cand scan', total=len(pair_overlap)):
        base = (t1, c1, t2, c2)
        if base in pos_set:
            continue
        try:
            nuniq_A = _get_nuniq(t1, c1)
            nuniq_B = _get_nuniq(t2, c2)
            if nuniq_A == 0 or nuniq_B == 0:
                continue
            if (is_high_overlap_hard_negative(ov) or
                is_low_cardinality_trap_negative(ov, nuniq_A, nuniq_B)):
                hard_candidates.append(base)
        except Exception:
            continue

    random.shuffle(hard_candidates)
    target_hard_neg = int(0.30 * target_neg)
    hard_added = 0
    print(f'[HARD NEG] candidates={len(hard_candidates)}, target={target_hard_neg}')

    final_dataset = []
    for f1, f2, label, pairkey in tqdm(raw_dataset, desc='raw→final'):
        base = (pairkey[0], pairkey[1], pairkey[2], pairkey[3]) if len(pairkey) >= 4 else pairkey
        ov = pair_overlap.get(base, [0.0] * 20)
        final_dataset.append((f1, f2, ov, label, base))

    for _hi, (t1, c1, t2, c2) in enumerate(tqdm(hard_candidates[:target_hard_neg], desc='hard neg add')):
        pairkey = (t1, c1, t2, c2, 'hardneg')
        if pairkey in existing_keys:
            continue
        try:
            df1, df2 = get_df(t1), get_df(t2)
            if c1 not in df1.columns or c2 not in df2.columns:
                continue
            f1 = get_feat_for_column(df1, c1)
            f2 = get_feat_for_column(df2, c2)
            ov = pair_overlap[(t1, c1, t2, c2)]
            final_dataset.append((f1, f2, ov, 0, (t1, c1, t2, c2)))
            existing_keys.add(pairkey)
            hard_added += 1
        except Exception:
            continue
        if (_hi + 1) % 50 == 0:
            csv_cache.clear()
            gc.collect()
    csv_cache.clear(); gc.collect()
    print(f'hard neg added: {hard_added}')

    save_ckpt(_ckpt_name, {
        'final_dataset': final_dataset,
        'existing_keys': existing_keys,
        'pair_overlap':  pair_overlap,
        'hard_added':    hard_added,
    })

clear_memory()


In [ ]:
# [v31 cell 15d] Catalog scan + col_state 사전 계산 + 체크포인트
_ckpt_name = 'col_state_catalog'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    all_columns = _loaded['all_columns']
    col_state   = _loaded['col_state']
    print(f'  all_columns: {len(all_columns)}, col_state size: {len(col_state)}')
else:
    print('🚀 Scanning catalog + precomputing col_state...')
    import math as _math
    from collections import Counter as _Counter_cs

    def _col_stats(series):
        s = series.dropna().astype(str)
        if len(s) == 0:
            return None
        n = len(s)
        sample = s.sample(min(200, n), random_state=0) if n > 200 else s
        lens = sample.str.len()
        mean_len = float(lens.mean()) if len(lens) > 0 else 0.0
        total_chars = max(int(lens.sum()), 1)
        alpha_chars = int(sample.str.count(r'[A-Za-z]').sum())
        digit_chars = int(sample.str.count(r'[0-9]').sum())
        alpha_ratio = alpha_chars / total_chars
        digit_ratio = digit_chars / total_chars
        nuniq = s.nunique()
        return mean_len, alpha_ratio, digit_ratio, nuniq

    all_columns = []
    for fname in tqdm(os.listdir(dataset_path), desc='catalog scan'):
        if not fname.endswith('.csv'):
            continue
        try:
            tname = fname.replace('.csv', '')
            df_cc = get_df(tname)
            for c in df_cc.columns:
                s = df_cc[c].dropna()
                if len(s) < 50:
                    continue
                stat = _col_stats(s)
                if stat is None:
                    continue
                mean_len, alpha_r, digit_r, nu = stat
                all_columns.append((tname, c, mean_len, alpha_r, digit_r, nu))
        except Exception:
            continue
    print(f'  catalog columns: {len(all_columns)}')

    # col_state 사전 계산
    print('[col_state] precomputing per-column derivatives...')

    def _cc_v30(n):
        if n <= 2: return 0
        if n <= 20: return 1
        if n <= 200: return 2
        return 3

    col_state = {}
    for _cs_i, (t_cs, c_cs, _ml, _ar, _dr, _nu) in enumerate(tqdm(all_columns, desc='col_state')):
        try:
            df_cs = get_df(t_cs)
            if c_cs not in df_cs.columns:
                continue
            s_cs = df_cs[c_cs].dropna().astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
            s_list = list(s_cs)
            s_set  = set(s_list)
            vc_cs = s_cs.value_counts()
            n_raw_cs = len(s_list); nuniq_cs = len(s_set)
            top10 = list(vc_cs.head(10).values / max(n_raw_cs, 1)) if len(vc_cs) > 0 else []
            while len(top10) < 10: top10.append(0.0)
            top1 = float(vc_cs.iloc[0] / max(n_raw_cs, 1)) if len(vc_cs) > 0 else 0.0
            sample_cs = s_list[:500]
            bigrams = []
            for v_cs in sample_cs:
                for k_cs in range(len(v_cs) - 1):
                    bigrams.append(v_cs[k_cs:k_cs+2])
            if bigrams:
                cnt_cs = _Counter_cs(bigrams)
                total_cs = sum(cnt_cs.values())
                probs_cs = [c_ / total_cs for c_ in cnt_cs.values()]
                H_cs = -sum(p * _math.log(p + 1e-9) for p in probs_cs)
                ent_cs = H_cs / _math.log(len(probs_cs) + 1)
            else:
                ent_cs = 0.0
            col_state[(t_cs, c_cs)] = {
                'set': s_set, 'n_raw': n_raw_cs, 'nuniq': nuniq_cs,
                'top10': top10, 'top1': top1, 'ent': ent_cs, 'cc': _cc_v30(nuniq_cs),
                'unique_ratio': nuniq_cs / max(n_raw_cs, 1),
            }
        except Exception:
            continue
        if (_cs_i + 1) % 50 == 0:
            csv_cache.clear(); gc.collect()
    csv_cache.clear(); gc.collect()
    print(f'  col_state size: {len(col_state)}')

    save_ckpt(_ckpt_name, {
        'all_columns': all_columns,
        'col_state':   col_state,
    })

clear_memory()


In [ ]:
# [v31 cell 15e] 합성 hard negative trap 4종 (trap4 신규: numeric narrow-range) + 체크포인트
_ckpt_name = 'final_dataset_step2'
_loaded = load_ckpt(_ckpt_name) if not globals().get('FORCE_RECOMPUTE', False) else None

if _loaded is not None:
    print(f'📦 Loaded from checkpoint: {ckpt_path(_ckpt_name)}')
    final_dataset = _loaded['final_dataset']
    existing_keys = _loaded['existing_keys']
    pair_overlap  = _loaded['pair_overlap']
    print(f'  final_dataset size: {len(final_dataset)}')
else:
    print('🚀 Generating 4-type synthetic traps (medium-card + name-like + code-like + numeric-narrow)...')
    import math as _math

    if 'col_state' not in locals():
        prev = load_ckpt('col_state_catalog')
        all_columns = prev['all_columns']
        col_state   = prev['col_state']
    if 'final_dataset' not in locals():
        prev = load_ckpt('final_dataset_step1')
        final_dataset = prev['final_dataset']
        existing_keys = prev['existing_keys']
        pair_overlap  = prev['pair_overlap']

    def _overlap_from_state(sA, sB):
        inter = sA['set'] & sB['set']; union = sA['set'] | sB['set']
        inter_size, union_size = len(inter), len(union)
        nA, nB = sA['nuniq'], sB['nuniq']
        nA_raw, nB_raw = sA['n_raw'], sB['n_raw']
        if nA_raw == 0 or nB_raw == 0 or not sA['set'] or not sB['set']:
            return [0.0] * 20
        jaccard = inter_size / union_size if union_size > 0 else 0.0
        con_a   = inter_size / nA if nA > 0 else 0.0
        con_b   = inter_size / nB if nB > 0 else 0.0
        containment_max = max(con_a, con_b); containment_min = min(con_a, con_b)
        containment_gap = abs(con_a - con_b)
        fk_direction_score = con_b - con_a
        one_sided_full = 1.0 if (con_a >= 0.999 or con_b >= 0.999) else 0.0
        max_ur = max(sA['unique_ratio'], sB['unique_ratio'])
        min_ur = min(sA['unique_ratio'], sB['unique_ratio'])
        intersection_size_ratio = inter_size / max(min(nA, nB), 1)
        union_size_ratio        = union_size / max(nA + nB, 1)
        both_low_unique_flag = 1.0 if (max_ur < 0.5) else 0.0
        low_cardinality_flag = 1.0 if (min(nA, nB) <= 3) else 0.0
        log_card_ratio = _math.log(nA + 1) - _math.log(nB + 1)
        freq_A = nA_raw / max(nA, 1); freq_B = nB_raw / max(nB, 1)
        log_freq_ratio = _math.log(freq_A + 1) - _math.log(freq_B + 1)
        freq_dist_diff = sum(abs(sA['top10'][i] - sB['top10'][i]) for i in range(10))
        bigram_entropy_diff = abs(sA['ent'] - sB['ent'])
        top1_concentration_diff = abs(sA['top1'] - sB['top1'])
        cardinality_class_match = 1.0 if sA['cc'] == sB['cc'] else 0.0
        return [float(jaccard), float(con_a), float(con_b),
                float(containment_max), float(containment_min), float(containment_gap),
                float(fk_direction_score), float(one_sided_full),
                float(intersection_size_ratio), float(union_size_ratio),
                float(max_ur), float(min_ur),
                float(both_low_unique_flag), float(low_cardinality_flag),
                float(log_card_ratio), float(log_freq_ratio),
                float(freq_dist_diff), float(bigram_entropy_diff),
                float(top1_concentration_diff), float(cardinality_class_match)]

    def _try_add_synth(t1, c1, t2, c2, tag, trap_kind):
        if t1 == t2: return False
        base = (t1, c1, t2, c2)
        if base in pos_set: return False
        pairkey = (t1, c1, t2, c2, tag)
        if pairkey in existing_keys: return False
        sA = col_state.get((t1, c1)); sB = col_state.get((t2, c2))
        if sA is None or sB is None: return False
        ov = _overlap_from_state(sA, sB)
        jacc = ov[0]; con_max = ov[3]; gap = ov[5]
        ok = False
        if   trap_kind == 'medium_card':    ok = (jacc >= 0.5 and con_max >= 0.7 and gap <= 0.30)
        elif trap_kind == 'name_like':      ok = (jacc >= 0.3 and con_max >= 0.5)
        elif trap_kind == 'code_like':      ok = (jacc >= 0.4 and con_max >= 0.6)
        elif trap_kind == 'numeric_narrow': ok = (jacc >= 0.25 and con_max >= 0.5 and gap <= 0.40)
        elif trap_kind == 'medium_numeric_partial': ok = (jacc >= 0.2 and jacc <= 0.6 and con_max >= 0.5 and con_max < 0.9 and gap <= 0.40)
        if not ok: return False
        try:
            df1 = get_df(t1); df2 = get_df(t2)
            if c1 not in df1.columns or c2 not in df2.columns:
                return False
            feat1 = get_feat_for_column(df1, c1)
            feat2 = get_feat_for_column(df2, c2)
        except Exception:
            return False
        pair_overlap[base] = ov
        pair_overlap[(t2, c2, t1, c1)] = _reverse_overlap_v30(ov)
        final_dataset.append((feat1, feat2, ov, 0, base))
        existing_keys.add(pairkey)
        return True

    # Trap 1: medium-cardinality
    trap1_cols = [r for r in all_columns if 5 <= r[5] <= 30]
    random.seed(42); random.shuffle(trap1_cols)
    SYNTH1_TARGET = 70; synth1 = 0
    print(f'  [trap1 medium-card] candidates={len(trap1_cols)}, target={SYNTH1_TARGET}')
    for i in tqdm(range(len(trap1_cols)), desc='trap1 medium-card'):
        if synth1 >= SYNTH1_TARGET: break
        t1, c1 = trap1_cols[i][0], trap1_cols[i][1]
        for j in range(i + 1, len(trap1_cols)):
            if synth1 >= SYNTH1_TARGET: break
            t2, c2 = trap1_cols[j][0], trap1_cols[j][1]
            if _try_add_synth(t1, c1, t2, c2, 'synth_med_card', 'medium_card'):
                synth1 += 1
    print(f'  trap1 added: {synth1}')
    csv_cache.clear(); gc.collect()

    # Trap 2: name-like
    trap2_cols = [r for r in all_columns if r[2] >= 8 and r[3] >= 0.6]
    random.seed(43); random.shuffle(trap2_cols)
    SYNTH2_TARGET = 70; synth2 = 0
    print(f'  [trap2 name-like] candidates={len(trap2_cols)}, target={SYNTH2_TARGET}')
    for i in tqdm(range(len(trap2_cols)), desc='trap2 name-like'):
        if synth2 >= SYNTH2_TARGET: break
        t1, c1 = trap2_cols[i][0], trap2_cols[i][1]
        for j in range(i + 1, len(trap2_cols)):
            if synth2 >= SYNTH2_TARGET: break
            t2, c2 = trap2_cols[j][0], trap2_cols[j][1]
            if _try_add_synth(t1, c1, t2, c2, 'synth_name_like', 'name_like'):
                synth2 += 1
    print(f'  trap2 added: {synth2}')
    csv_cache.clear(); gc.collect()

    # Trap 3: code-like
    trap3_cols = [r for r in all_columns if 4 <= r[2] <= 14 and r[4] >= 0.5]
    random.seed(44); random.shuffle(trap3_cols)
    SYNTH3_TARGET = 60; synth3 = 0
    print(f'  [trap3 code-like] candidates={len(trap3_cols)}, target={SYNTH3_TARGET}')
    for i in tqdm(range(len(trap3_cols)), desc='trap3 code-like'):
        if synth3 >= SYNTH3_TARGET: break
        t1, c1 = trap3_cols[i][0], trap3_cols[i][1]
        for j in range(i + 1, len(trap3_cols)):
            if synth3 >= SYNTH3_TARGET: break
            t2, c2 = trap3_cols[j][0], trap3_cols[j][1]
            if _try_add_synth(t1, c1, t2, c2, 'synth_code_like', 'code_like'):
                synth3 += 1
    print(f'  trap3 added: {synth3}')
    csv_cache.clear(); gc.collect()

    # Trap 4 (v31 신규): numeric narrow-range (hero-stat / small-ID 패턴)
    trap4_cols = [r for r in all_columns if r[4] >= 0.8 and r[2] <= 4 and r[5] <= 200]
    random.seed(45); random.shuffle(trap4_cols)
    SYNTH4_TARGET = 60; synth4 = 0
    print(f'  [trap4 numeric-narrow] candidates={len(trap4_cols)}, target={SYNTH4_TARGET}')
    for i in tqdm(range(len(trap4_cols)), desc='trap4 numeric-narrow'):
        if synth4 >= SYNTH4_TARGET: break
        t1, c1 = trap4_cols[i][0], trap4_cols[i][1]
        for j in range(i + 1, len(trap4_cols)):
            if synth4 >= SYNTH4_TARGET: break
            t2, c2 = trap4_cols[j][0], trap4_cols[j][1]
            if _try_add_synth(t1, c1, t2, c2, 'synth_num_narrow', 'numeric_narrow'):
                synth4 += 1
    print(f'  trap4 added: {synth4}')
    csv_cache.clear(); gc.collect()

    # Trap 5 (v33 신규): medium-numeric-id — superhero hero_attribute_value ↔ id 패턴
    # 양쪽 nuniq 30~200 (medium), digit_ratio 높음, jaccard 낮으나 containment 큼 (부분 포함)
    trap5_cols = [r for r in all_columns if 30 <= r[5] <= 150 and r[4] >= 0.7 and r[2] <= 4]
    random.seed(46); random.shuffle(trap5_cols)
    SYNTH5_TARGET = 80; synth5 = 0
    print(f'  [trap5 medium-numeric-partial] candidates={len(trap5_cols)}, target={SYNTH5_TARGET}')
    for i in tqdm(range(len(trap5_cols)), desc='trap5 medium-numeric-partial'):
        if synth5 >= SYNTH5_TARGET: break
        t1, c1 = trap5_cols[i][0], trap5_cols[i][1]
        for j in range(i + 1, len(trap5_cols)):
            if synth5 >= SYNTH5_TARGET: break
            t2, c2 = trap5_cols[j][0], trap5_cols[j][1]
            if _try_add_synth(t1, c1, t2, c2, 'synth_med_num_partial', 'medium_numeric_partial'):
                synth5 += 1
    print(f'  trap5 added: {synth5}')
    csv_cache.clear(); gc.collect()

    total_synth = synth1 + synth2 + synth3 + synth4 + synth5
    print(f'[SYNTH HARD NEG v33] total added: {total_synth} (trap1={synth1}, trap2={synth2}, trap3={synth3}, trap4={synth4}, trap5={synth5})')

    save_ckpt(_ckpt_name, {
        'final_dataset': final_dataset,
        'existing_keys': existing_keys,
        'pair_overlap':  pair_overlap,
    })

clear_memory()


In [ ]:
# [v31 cell 15f] final_dataset 셔플 + 요약
if 'final_dataset' not in locals():
    prev = load_ckpt('final_dataset_step2')
    assert prev is not None, '15e 체크포인트 없음'
    final_dataset = prev['final_dataset']
    existing_keys = prev['existing_keys']
    pair_overlap  = prev['pair_overlap']

if 'raw_dataset' in locals():
    del raw_dataset
clear_memory()

random.seed(42)
random.shuffle(final_dataset)

n_pos = sum(1 for x in final_dataset if x[3] == 1)
n_neg = sum(1 for x in final_dataset if x[3] == 0)
print(f'\n=== final_dataset: pos={n_pos}, neg={n_neg}, total={len(final_dataset)} ===')


#### 7. 모델 학습 데이터셋 저장 (final_dataset)

**final_dataset 구성**
- 컬럼1 특징 (42차원)
- 컬럼2 특징 (42차원)
- Overlap 특징 (4차원)
- 라벨 (0 또는 1)
- 컬럼쌍 정보

---

**사용 시나리오**

<small>**시나리오 1: 피처 고정**</small>
- ✓ 모델 구조 변경 시 직접 사용
- ✓ 캐시된 데이터 재활용

<small>**시나리오 2: 피처 변경 필요**</small>
- → **5단계부터 재수행 필수**
  - 5. 데이터 수집 및 특징 추출
  - 6. 학습용 라벨링 및 Dataset 생성
  - 7. final_dataset 업데이트


In [ ]:
# [cell7]
import pickle
import os

# 1. 저장 경로 설정
# base_path는 기존 설정대로 사용하시되, 파일명에 'augmented'를 붙여 구분하는 것이 좋습니다.
# augmented_data_path = os.path.join(base_path, 'augmented_train_data.pkl')
# [cell1에서 정의]

# 2. 저장할 데이터 구성
# final_dataset: [(feat1, feat2, ov_feat, label), ...] 구조
data_to_save = {
    'final_dataset': final_dataset,  # 증강된 피처 쌍 + Overlap + 라벨 통합본
    'metadata_info': {
        # v13: source-specific augmentation (no single AUGMENT_MULTIPLIER)
        'ratio_tpcds':  RATIO_TPC_DS,
        'ratio_tpch':   RATIO_TPC_H,
        'ratio_spider': RATIO_SPIDER,
        'target_total_pos': TARGET_TOTAL_POS,
        'sampling_frac': SAMPLING_FRAC,
        'total_samples': len(final_dataset),
    }
}

# 3. Pickle 파일로 저장
try:
    with open(augmented_data_path, 'wb') as f:
        # 데이터가 클 수 있으므로 protocol=4 이상 권장 (파이썬 3.4+ 기본)
        pickle.dump(data_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"✅ 증강 전처리 데이터 저장 완료!")
    print(f"📍 경로: {augmented_data_path}")
    print(f"📊 총 샘플 수: {len(final_dataset)}개 (Positive + Negative)")
    print(f"💡 이제 재실행 시 데이터 로드 셀만 실행하면 바로 모델 학습이 가능합니다.")

except Exception as e:
    print(f"❌ 저장 중 에러 발생: {e}")
    print("드라이브 마운트 상태나 용량, 경로 권한을 확인해 주세요.")


# # 재수행시
# # (예시) 데이터 로드 및 확인 코드
# with open(augmented_data_path, 'rb') as f:
#     loaded_data = pickle.load(f)

# # 데이터셋 복구
# final_dataset = loaded_data['final_dataset']

# # 샘플 하나 확인 (제대로 로드됐는지)
# sample = final_dataset[0]
# print(f"로드된 샘플 피처 A 차원: {sample[0].shape}")
# print(f"로드된 샘플 피처 B 차원: {sample[1].shape}")
# print(f"로드된 샘플 Overlap 피처: {sample[2]}")
# print(f"로드된 샘플 라벨: {sample[3]}")

#### 9. 데이터 스케일링

##### 테이블 단위 분리 (Train/Test Split)

<small>**목표: 테스트 테이블은 학습에서 완전히 배제 → 실제 일반화 성능 측정**</small>

- TEST_TABLES에 포함된 테이블과 연관된 컬럼 쌍 제외
- Train/Test 완전 분리


final_dataset
│
├─ TEST_TABLES 포함 여부로 분리
│   ├─ train_samples
│   └─ test_samples
│
├─ train 데이터로만 Scaler 학습
│
└─ train/test 모두 Scaler 적용
├─ train_samples_scaled
└─ test_samples_scaled


---

##### 테이블 단위 분리 관련 주요 논문

| 논문 | Venue | 내용 |
|------|-------|------|
| DODUO (Suhara et al., 2022) | VLDB | 컬럼 타입 탐지, 테이블 단위 hold-out 명시 |
| Starmie (Fan et al., 2023) | VLDB | 컬럼 표현 학습, unseen table 평가 |
| Valentine (Koutras et al., 2021) | ICDE | 스키마 매칭 벤치마크, 테이블 단위 평가 |
| ALITE (Khatiwada et al., 2022) | VLDB | 테이블 보완, hold-out 테이블 설정 |

---

##### 스케일러 선택 과정

**StandardScaler (최종 선택)**
- 각 피처를 평균 0, 표준편차 1로 정규화
- 컬럼 통계 피처의 이상치 및 불균일한 범위 처리

**시도: RobustScaler** (결과 미흡)
- 중앙값과 사분위수 범위(IQR) 기반 정규화
- 이상치 영향 최소화
- 논문: Huber, P.J. (1981). Robust Statistics. Wiley.
- **결과**: 성능 저하 → 원래대로 StandardScaler 사용

---

##### 스케일러 분리 사용 이유

<small>**컬럼 피처 vs 오버랩 피처는 성격이 완전히 다릅니다**</small>

| 항목 | 컬럼 개별 피처 | 오버랩 피처 |
|------|-------------|----------|
| **대상** | 컬럼 하나의 통계 | 두 컬럼 간 관계값 |
| **예시** | entropy, null_ratio, unique_count | Jaccard, Containment A/B, OV_Flag |
| **값 범위** | 다양하고 넓음 | 대부분 0~1 (비율) |
| **학습 시점** | 컬럼 단위로 fit | 쌍(pair) 단위로 fit |
| **스케일러** | StandardScaler | MinMaxScaler |

**분리의 이점**
- 같은 스케일러로 함께 fit하면 서로 다른 분포 혼재 → 정규화 기준 왜곡
- 분리하면 각 피처 그룹의 특성에 맞는 정규화 기준 독립 유지

---

##### 최종 정규화 방식

- **컬럼 피처** (42차원) → **StandardScaler**
- **오버랩 피처** (4차원: Jaccard, Containment A/B, OV_Flag) → **MinMaxScaler** (이미 0~1 범위)


<small>컬럼 개별 피처	오버랩 피처
대상	컬럼 하나의 통계	두 컬럼 간 관계값
예시	entropy, null_ratio, unique_count...	Jaccard, Containment A/B, OV_Flag
값 범위	다양하고 넓음	대부분 0~1 (비율)
학습 시점	컬럼 단위 데이터로 fit	쌍(pair) 단위 데이터로 fit
같은 스케일러로 함께 fit하면 서로 다른 분포가 섞여서 정규화 기준이 왜곡됩니다. 분리하면 각 피처 그룹의 특성에 맞는 정규화 기준을 독립적으로 유지할 수 있습니다.

오버랩 피처(Jaccard, Containment A/B, OV_Flag)는 이미 0~1 사이의 비율값 -> MinMaxScaler</small>

In [ ]:
# [extra] pos:neg = 1:2 로 맞춘 뒤 그대로 train/test split에 사용

import random

TARGET_POS_TOTAL = 700
NEG_PER_POS = 2
TARGET_NEG_TOTAL = TARGET_POS_TOTAL * NEG_PER_POS
random.seed(42)

positive_samples = [item for item in final_dataset if item[3] == 1]
negative_samples = [item for item in final_dataset if item[3] == 0]

print(f"원본 final_dataset positive: {len(positive_samples)}")
print(f"원본 final_dataset negative: {len(negative_samples)}")

if len(positive_samples) == 0:
    raise ValueError('positive sample이 없습니다.')
if len(negative_samples) == 0:
    raise ValueError('negative sample이 없습니다.')

# 1) positive를 700개로 맞춤
if len(positive_samples) < TARGET_POS_TOTAL:
    needed_pos = TARGET_POS_TOTAL - len(positive_samples)
    extra_positive = [random.choice(positive_samples) for _ in range(needed_pos)]
    positive_samples_balanced = positive_samples + extra_positive
else:
    positive_samples_balanced = random.sample(positive_samples, TARGET_POS_TOTAL)

# 2) negative를 positive의 2배(1:2)로 맞춤
# v16: hard negative를 우선 보존 (random sampling으로 손실되지 않게)
def _is_hardneg(sample):
    pair_key = sample[-1] if len(sample) >= 5 else None
    if isinstance(pair_key, tuple):
        return any('hardneg' in str(p) for p in pair_key)
    return False

if len(negative_samples) < TARGET_NEG_TOTAL:
    needed_neg = TARGET_NEG_TOTAL - len(negative_samples)
    extra_negative = [random.choice(negative_samples) for _ in range(needed_neg)]
    negative_samples_balanced = negative_samples + extra_negative
else:
    # hard neg 모두 보존 + 부족분 random fill
    hard_negs_kept = [s for s in negative_samples if _is_hardneg(s)]
    rand_negs_pool = [s for s in negative_samples if not _is_hardneg(s)]
    rand_needed = max(0, TARGET_NEG_TOTAL - len(hard_negs_kept))
    rand_kept = random.sample(rand_negs_pool, min(rand_needed, len(rand_negs_pool)))
    negative_samples_balanced = hard_negs_kept + rand_kept
    if len(negative_samples_balanced) > TARGET_NEG_TOTAL:
        negative_samples_balanced = random.sample(negative_samples_balanced, TARGET_NEG_TOTAL)
    print(f'  → hard neg preserved: {len(hard_negs_kept)}, random neg added: {len(rand_kept)}')

final_dataset = positive_samples_balanced + negative_samples_balanced
random.shuffle(final_dataset)

final_pos = sum(1 for x in final_dataset if x[3] == 1)
final_neg = sum(1 for x in final_dataset if x[3] == 0)

print(f"조정 후 final_dataset positive: {final_pos}")
print(f"조정 후 final_dataset negative: {final_neg}")
print(f"조정 후 final_dataset total   : {len(final_dataset)}")
print(f"최종 비율 pos:neg = 1:{final_neg/final_pos:.1f}")
print('이 final_dataset이 그대로 train/test split에 들어갑니다.')


In [ ]:
import gc
# [cell9] train/test split using fixed pos:neg = 1:2 dataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import numpy as np
from tqdm.auto import tqdm

TEST_TABLES = {'store_returns', 'inventory'}

train_samples = []
test_samples = []

for item in tqdm(final_dataset, desc="split prep"):
    f1, f2, ov, label, pair_key = item
    t1, c1, t2, c2 = pair_key

    table_i = str(t1).replace('.csv', '').lower()
    table_j = str(t2).replace('.csv', '').lower()

    if table_i in TEST_TABLES or table_j in TEST_TABLES:
        test_samples.append((f1, f2, ov, label, pair_key))
    else:
        train_samples.append((f1, f2, ov, label, pair_key))

train_features_for_scaler = []
for f1, f2, _, _, _ in tqdm(train_samples, desc="scaler fit"):
    train_features_for_scaler.append(np.array(f1).flatten())
    train_features_for_scaler.append(np.array(f2).flatten())

scaler = StandardScaler()
scaler.fit(np.vstack(train_features_for_scaler))

scaled_train = scaler.transform(np.vstack(train_features_for_scaler))
clip_lower = np.percentile(scaled_train, 1, axis=0)
clip_upper = np.percentile(scaled_train, 99, axis=0)
joblib.dump((clip_lower, clip_upper), clip_bounds_path)
joblib.dump(scaler, scaler_path)

train_ov = np.array([np.array(ov) for _, _, ov, _, _ in train_samples])
ov_scaler = MinMaxScaler()
ov_scaler.fit(train_ov)
joblib.dump(ov_scaler, ov_scaler_path)

def scale_sample(sample, scaler, ov_scaler, clip_lower, clip_upper):
    f1, f2, ov, label, pair_key = sample
    f1_s = scaler.transform(np.array(f1).reshape(1, -1)).flatten()
    f2_s = scaler.transform(np.array(f2).reshape(1, -1)).flatten()
    ov_s = np.array(ov).flatten().tolist()
    return (f1_s, f2_s, ov_s, label, pair_key)

train_samples_scaled = [scale_sample(s, scaler, ov_scaler, clip_lower, clip_upper) for s in train_samples]
test_samples_scaled  = [scale_sample(s, scaler, ov_scaler, clip_lower, clip_upper) for s in test_samples]

num_train_pos = sum(1 for *_, label, __ in train_samples_scaled if label == 1)
num_train_neg = sum(1 for *_, label, __ in train_samples_scaled if label == 0)
num_test_pos  = sum(1 for *_, label, __ in test_samples_scaled if label == 1)
num_test_neg  = sum(1 for *_, label, __ in test_samples_scaled if label == 0)

print(f'📍 최종 샘플 피처 차원: {np.concatenate([train_samples_scaled[0][0], train_samples_scaled[0][1], np.array(train_samples_scaled[0][2])]).shape}')
print('-' * 50)
print(f'✅ pos:neg=1:2 데이터셋 구성 후 테이블 단위 분리 완료 (Test 테이블: {sorted(TEST_TABLES)})')
print(f'📏 피처 최종 차원: {len(train_samples_scaled[0][0]) + len(train_samples_scaled[0][1]) + len(train_samples_scaled[0][2])}')
print(f'📊 Train samples: {len(train_samples_scaled)}')
print(f'📊 Test samples : {len(test_samples_scaled)}')
print(f'📊 Train Positive: {num_train_pos}')
print(f'📊 Train Negative: {num_train_neg}')
print(f'📊 Test Positive: {num_test_pos}')
print(f'📊 Test Negative: {num_test_neg}')
print(f'📍 Scaler 저장 경로: {scaler_path}')
print('-' * 50)

gc.collect()  # v21/v22: scaler fit 후 메모리 정리


#### 10. Dataset & Loss Function

##### 1. AugmentedPairDataset 클래스

<small>**특징 통합 및 PyTorch 변환 + 정답 라벨 반환**</small>

**기능**
- 두 컬럼의 특징(f1, f2)과 오버랩 피처(ov) 결합
- PyTorch Tensor로 변환
- 학습용 라벨 제공

**특징 결합 구조**
[u (42)] + [v (42)] + [|u-v| (42)] + [u*v (42)] + [ov (4)]
↓
combined_feat (172)


**주요 처리**
| 작업 | 설명 |
|------|------|
| u, v 추출 | f1, f2를 1D 배열로 변환 |
| 차이 계산 | `\|u - v\|` (원소별 절대값) |
| 원소곱 | `u * v` (원소별 곱셈) |
| Overlap 연결 | OV 피처 4차원 추가 |
| Tensor 변환 | FloatTensor(특징), LongTensor(라벨) |

---

##### 2. Focal Loss

<small>**클래스 불균형 문제 해결 (양성/음성 샘플 비율 조정)**</small>

**파라미터**
- `alpha_pos=0.7` | 양성 샘플(label=1) 가중치
- `alpha_neg=0.3` | 음성 샘플(label=0) 가중치
- `gamma=1.5` | Focusing parameter (어려운 샘플에 집중)

**손실함수 계산 흐름**
Cross Entropy Loss 계산
↓
가능도(probability) 계산: pt = exp(-ce_loss)
↓
라벨별 alpha 값 할당
label=1 → alpha_pos(0.7)
label=0 → alpha_neg(0.3)
↓
Focal Loss = alpha_t × (1-pt)^gamma × ce_loss
↓
배치 평균


**수식**

FL(pt) = -α_t × (1 - pt)^γ × log(pt)

- **α_t**: 클래스별 가중치 (양성/음성 불균형 조정)
- **(1 - pt)^γ**: Focusing term (쉬운 샘플 억제, 어려운 샘플 강조)

**효과**
- 음성 샘플이 많을 때 양성 샘플에 더 집중
- Gamma 값이 클수록 어려운 샘플에 더 높은 손실
- 클래스 불균형 상황에서 모델 성능 향상





모델 초기화 및 학습
Dataset 정의 → FocalLoss 정의 → 모델 초기화 → DataLoader → 학습 루프 → 최적 모델 저장

클래스 불균형 문제를 해결하기 위한 손실 함수로 Focalloss사용
    alpha_pos=0.85, alpha_neg=0.15  # Positive에 더 큰 가중치 -> 0.5로 변경해서 테스트예정
    gamma=1.5                        # 쉬운 샘플 가중치 감소
이미 positive:negative를 1:1로 수정해서 일반 CrossEntropy + gamma만 쓰는 방식도 고려 (X 성능이 더 안나옴)

Train/Validation dataset = 80:20

In [ ]:
# [v30 cell 10] AugmentedPairDataset + BCE Loss
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import copy
import numpy as np


class AugmentedPairDataset(Dataset):
    """v30: 98-dim input
    layout: [u_stat(18), u_char(18), v_stat(18), v_char(18), pair_stat(6), overlap(20)] = 98-dim
    """

    EPS = 1e-6

    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def _pair_stat(self, u, v):
        """Derive 6 pair_stat features from 36-dim u/v feature vectors.
        Uses stat indices (u[0:18] is stat):
          [0:n_rows(log), 1:null_ratio, 2:unique_ratio, 3:dup_ratio,
           4:avg_freq, 5:median_freq, 6:min_max_ratio, 7:range_per_med(tanh),
           8:std_per_mean(tanh), 9:skew, 10:kurt, 11:entropy, ...]
        """
        ratio_diff       = abs(u[2] - v[2])
        unique_ratio_diff= u[2] - v[2]
        null_ratio_diff  = u[1] - v[1]
        range_ratio      = u[7] / (abs(v[7]) + self.EPS)
        std_ratio        = u[8] / (abs(v[8]) + self.EPS)
        entropy_ratio    = u[11] / (abs(v[11]) + self.EPS)
        range_ratio   = np.tanh(range_ratio - 1.0)
        std_ratio     = np.tanh(std_ratio - 1.0)
        entropy_ratio = np.tanh(entropy_ratio - 1.0)
        return np.array([ratio_diff, unique_ratio_diff, null_ratio_diff,
                         range_ratio, std_ratio, entropy_ratio], dtype=np.float32)

    def __getitem__(self, idx):
        f1, f2, ov, label, *_ = self.samples[idx]
        u = np.array(f1, dtype=np.float32).flatten()  # 36-dim
        v = np.array(f2, dtype=np.float32).flatten()  # 36-dim
        ov_feat = np.array(ov, dtype=np.float32) if not isinstance(ov, (int, float)) else np.zeros(20, dtype=np.float32)

        # Pad/truncate to 36 if necessary
        if len(u) < 36:
            u = np.concatenate([u, np.zeros(36 - len(u), dtype=np.float32)])
        if len(v) < 36:
            v = np.concatenate([v, np.zeros(36 - len(v), dtype=np.float32)])
        if len(ov_feat) < 20:
            ov_feat = np.concatenate([ov_feat, np.zeros(20 - len(ov_feat), dtype=np.float32)])
        elif len(ov_feat) > 20:
            ov_feat = ov_feat[:20]

        u_stat, u_char = u[:18], u[18:36]
        v_stat, v_char = v[:18], v[18:36]
        pair_stat = self._pair_stat(u, v)

        combined = np.concatenate([u_stat, u_char, v_stat, v_char, pair_stat, ov_feat])  # 98-dim
        return torch.FloatTensor(combined), torch.tensor(float(label), dtype=torch.float32)


# v13: BCE with logits (사용자 spec)
# 기존 FocalLoss는 alias로 BCE 래핑 (다른 셀이 FocalLoss 참조해도 동작)
class FocalLoss(nn.Module):
    """v34_final_ablation (정정): BCE wrapper — v34_final과 동일 protocol"""
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.criterion = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        if logits.ndim > 1:
            logits = logits.squeeze(-1)
        targets = targets.float()
        return self.criterion(logits, targets)


In [ ]:
# [extra] Dataset size summary
print("-" * 50)
print("Dataset Size Summary")
print(f"Train samples: {len(train_samples_scaled)}")
print(f"Test samples : {len(test_samples_scaled)}")

# Positive / Negative label counts (train)
train_labels = [lbl for (_, _, _, lbl, _) in train_samples_scaled]
print(f"Train Positive: {sum(train_labels)}")
print(f"Train Negative: {train_labels.count(0)}")

# Positive / Negative label counts (test)
test_labels = [lbl for (_, _, _, lbl, _) in test_samples_scaled]
print(f"Test Positive: {sum(test_labels)}")
print(f"Test Negative: {test_labels.count(0)}")
print("-" * 50)

#### 11. 모델 학습 (Training)

##### 1. 모델 초기화

<small>**차원 자동 계산 & 디바이스 설정**</small>

| 항목 | 설명 |
|------|------|
| **Device** | GPU 사용 가능 시 cuda, 아니면 cpu |
| **입력 차원** | sample_f1으로부터 자동 계산 (1200) |
| **모델 선택** | CRIM_StatisticsOnly_Matcher (선택됨) |


device 설정
↓
train_samples_scaled[0][0] 샘플 추출
↓
flatten() → s_dim = 1200 자동 인식
↓
모델 초기화 & device로 이동


---

##### 2. 데이터 로더 설정

<small>**Train/Val Split & Batch 구성**</small>

| 항목 | 값 | 설명 |
|------|-----|------|
| **Train/Val Split** | 80:20 | Validation 크기: 전체의 20% |
| **배치 크기** | 64 | 한 번에 처리하는 샘플 수 |
| **Train 옵션** | shuffle=True | 매 epoch마다 데이터 섞기 |
| **Val 옵션** | shuffle=False | 일관성 있는 검증 |

---

##### 3. 손실함수 & 최적화기

**손실함수**
- `FocalLoss()` | 클래스 불균형 처리

**최적화기: Adam**
- `learning_rate = 0.0001` | 학습률
- `weight_decay = 1e-4` | L2 정규화 (과적합 방지)

**학습률 감소 스케줄러**
- `ReduceLROnPlateau` | val_loss 3 epoch 미개선 시 lr × 0.1

---

##### 4. 학습 루프

<small>**Epoch = 400, Patience = 10**</small>

for epoch in range(num_epochs):
<pre style="padding: 12px; border-radius: 5px; overflow-x: auto;">
for epoch in range(num_epochs):
    
    ┌─────────────────────────────┐
    │  TRAIN (model.train())      │
    ├─────────────────────────────┤
    │ for batch in train_loader:  │
    │   ├─ optimizer.zero_grad()  │ ① 기울기 초기화
    │   ├─ loss = criterion(...)  │ ② 순전파 & loss 계산
    │   ├─ loss.backward()        │ ③ 역전파 & 기울기 계산
    │   ├─ optimizer.step()       │ ④ 가중치 업데이트
    │   └─ total_tr_loss += ...   │ ⑤ loss 누적
    └─────────────────────────────┘
                │
                ↓
    ┌─────────────────────────────┐
    │  VALIDATION (eval())        │
    ├─────────────────────────────┤
    │ with torch.no_grad():       │ 기울기 계산 비활성화
    │   for batch in val_loader:  │
    │     total_val_loss += ...   │
    └─────────────────────────────┘
                │
                ↓
    ┌─────────────────────────────┐
    │  Early Stopping Check       │
    ├─────────────────────────────┤
    │ if avg_val < best_val_loss: │
    │   ✅ 최적 모델 저장             │
    │   patience_counter = 0      │
    │ else:                       │
    │   patience_counter += 1     │
    │   if patience ≥ 10: break   │ 조기 종료
    └─────────────────────────────┘
                │
                ↓
    scheduler.step(avg_val) → 학습률 조정
</pre>


조기 종료 (Early Stopping)

patience = 10 | 10 epoch 동안 개선 없으면 학습 중단
최적 모델 상태로 복원 후 저장


In [ ]:
# [v34_final_ablation] 7개 branch 조합 학습 loop — BCE + SWA per variant
#
# ★ Determinism 설정 적용됨 (cell 4 의 apply_seed 강화 + 이 셀의 DataLoader 패치) ★
# 목표: superhero/superhero_mask 같은 borderline 케이스가 많은 DB 에서
#       매 학습 실행마다 같은 가중치를 얻어 같은 metric 산출.
# 결과: 이 학습 루프로 저장된 stat+char+ov 체크포인트가
#       논문의 "Full CRIM" 단일 권위 모델로 사용 가능.
# 비용: 학습 속도 10~20% 느려짐, atomic op 일부 미지원 시 경고 (warn_only=True)
# ── 안전장치: post_fix 검증 (다른 노트북의 post_fix 로 v34 파일 덮어쓰기 방지) ──
assert 'post_fix' in dir(), 'cell 4 (post_fix 설정) 를 먼저 실행하세요.'
assert post_fix == '0517_v34_final_ablation', (
    f'⛔ post_fix="{post_fix}" — v34 원본 노트북의 기대값과 다름. '
    f'cell 4 를 다시 실행하세요. (예상값: 0517_v34_final_ablation)'
)
print(f'  ✓ post_fix={post_fix} 검증 통과')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import copy
import numpy as np
import os
import random
from tqdm.auto import tqdm

clear_memory()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_f1 = np.array(train_samples_scaled[0][0]).flatten()
s_dim = len(sample_f1)
print(f's_dim (column-level): {s_dim} (expected 36)')

# 데이터 로더 (SEED로 split 고정)
full_train_dataset = AugmentedPairDataset(train_samples_scaled)
val_size = max(1, int(len(full_train_dataset) * 0.2))
if SEED is not None:
    tr_sub, val_sub = torch.utils.data.random_split(
        full_train_dataset,
        [len(full_train_dataset) - val_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f'  random_split: seed={SEED} (reproducible)')
else:
    tr_sub, val_sub = torch.utils.data.random_split(
        full_train_dataset,
        [len(full_train_dataset) - val_size, val_size],
    )
    print(f'  random_split: NO seed (fully random)')

epoch_batch_size = 64

# ★ 결정론적 DataLoader ★
# - generator: 셔플 순서를 SEED 로 고정 → 매 epoch 같은 순서
# - worker_init_fn: num_workers > 0 일 때 각 worker 의 RNG 도 결정론적으로
# - num_workers=0 : 단일 프로세스가 가장 안전 (멀티 worker 는 OS 스케줄링 의존)
def _seed_worker(worker_id):
    """DataLoader worker 의 RNG 를 결정론적으로 초기화."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

_loader_gen = torch.Generator()
_loader_gen.manual_seed(SEED)

tr_loader = DataLoader(
    tr_sub, batch_size=epoch_batch_size, shuffle=True, drop_last=True,
    generator=_loader_gen,
    worker_init_fn=_seed_worker,
    num_workers=0,                   # 결정론 우선 (멀티 worker 비활성)
)
val_loader = DataLoader(
    val_sub, batch_size=epoch_batch_size, shuffle=False,
    num_workers=0,
)
print(f'train: {len(tr_sub)}, val: {len(val_sub)}')

# 첫 배치 input dim 확인
_xb, _yb = next(iter(tr_loader))
print(f'batch input dim: {_xb.shape[1]} (expected 98)')

ABLATION_CONFIGS = [
    ['stat'],
    ['char'],
    ['ov'],
    ['stat', 'char'],
    ['stat', 'ov'],
    ['char', 'ov'],
    ['stat', 'char', 'ov'],
]

ablation_paths = {}
ablation_logs  = {}

# SWA + 학습 hyperparams (v34_final과 동일)
learning_rate    = 0.0001
weight_decay_set = 1e-4
patience_num     = 4
num_epochs       = 150
swa_start_epoch  = 30
swa_lr           = 5e-5
swa_anneal_epochs = 5
patience         = 10   # SWA 시작 전에만 early stop

print(f'\n🎲 학습 시작 — SEED={SEED}, BCE + SWA (각 variant)')
print(f'  num_epochs={num_epochs}, swa_start_epoch={swa_start_epoch}, swa_lr={swa_lr}')

for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    print(f'\n=== Training ablation: {tag} (seed={SEED}, BCE+SWA) ===')

    # ★ 결정론적 학습 보장 ★
    # 매 variant 학습 직전에 SEED 를 다시 적용해야 함:
    #   - 이전 variant 학습에서 RNG 상태가 변경됨
    #   - 같은 SEED 로 RNG 를 리셋해야 7개 variant 모두 동일 출발점에서 학습
    #   - 이렇게 하면 stat+char+ov (Full) 행이 "main CRIM"으로 사용 가능한
    #     단일 권위 가중치가 됨 (논문에서 ablation full = main CRIM)
    apply_seed(SEED)

    # DataLoader generator 도 매 variant 마다 같은 seed 로 재초기화
    # (이전 variant 의 셔플로 generator 상태가 변경됐을 수 있음)
    _loader_gen.manual_seed(SEED)

    model = V33AblationMatcher(branch_config=cfg).to(device)
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)   # 실제로는 BCE wrapper
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay_set)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=patience_num)

    # SWA 설정 (variant별 새로)
    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=swa_lr, anneal_epochs=swa_anneal_epochs)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    train_losses, val_losses = [], []
    swa_active = False

    for epoch in range(num_epochs):
        model.train()
        total_tr_loss = 0
        for bx, by in tr_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_tr_loss += loss.item()

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                total_val_loss += criterion(model(bx), by).item()

        avg_tr = total_tr_loss / len(tr_loader)
        avg_val = total_val_loss / len(val_loader)
        train_losses.append(avg_tr); val_losses.append(avg_val)

        # SWA phase
        if epoch + 1 >= swa_start_epoch:
            if not swa_active:
                print(f'  🔄 SWA 시작 (epoch {epoch+1})')
                swa_active = True
            swa_scheduler.step()
            swa_model.update_parameters(model)
        else:
            scheduler.step(avg_val)

        if (epoch + 1) % 30 == 0 or epoch == 0:
            n_avg = swa_model.n_averaged.item() if swa_active else 0
            print(f'  Epoch [{epoch+1:3d}] Train: {avg_tr:.4f} | Val: {avg_val:.4f} | SWA_n={n_avg}')

        # SWA 진입 전에만 early stop
        if not swa_active:
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'  Early stop at epoch {epoch+1} (전 SWA 단계)')
                    break

    # 학습 후 처리
    swa_n_averaged = int(swa_model.n_averaged.item()) if swa_active else 0
    print(f'  학습 종료: SWA 평균 {swa_n_averaged} epoch')

    if swa_n_averaged >= 5:
        update_bn(tr_loader, swa_model, device=device)
        final_state = swa_model.module.state_dict()
        final_model_type = 'SWA'
    else:
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        final_state = model.state_dict()
        final_model_type = 'best_val'

    tag_safe = tag.replace('+', '_')
    save_path = os.path.join(base_path, f'multi_task_matching_model_{post_fix}_{tag_safe}.pth')
    torch.save({
        'model_state_dict': final_state,
        's_dim': s_dim,
        'branch_config': cfg,
        'seed': SEED,
        'focal_alpha': FOCAL_ALPHA,
        'focal_gamma': FOCAL_GAMMA,
        'swa_n_averaged': swa_n_averaged,
        'final_model_type': final_model_type,
        'best_val_loss': best_val_loss,
    }, save_path)
    ablation_paths[tag] = save_path
    ablation_logs[tag] = {'train_losses': train_losses, 'val_losses': val_losses, 'swa_n': swa_n_averaged}
    print(f'  ✅ Saved: {save_path}  (type={final_model_type}, SWA_n={swa_n_averaged})')
    clear_memory()

print(f'\n--- All 7 v34_final_ablation models trained (SEED={SEED}, BCE+SWA) ---')
print(f'paths: {list(ablation_paths.keys())}')


#### 12. 모델 평가 (Evaluation)

##### 1. 오버랩 브랜치 시각화 함수

<small>**visualize_gate_values: 모델의 Overlap 브랜치 반응성 확인**</small>

<pre style="padding: 12px; border-radius: 5px; overflow-x: auto;">
Overlap 브랜치 (4차원 입력)
    │
    ├─ 마지막 4차원 추출: x[:, -4:]
    │
    ├─ branch_overlap 통과
    │  └─ 64차원 출력
    │
    ├─ L2 Norm 계산 (강도)
    │  └─ strength = ||output||₂
    │
    └─ 시각화
       └─ scatter plot
          (OV_Flag vs Output Strength)
</pre>

**시각화 내용**
- X축: Overlap Flag 값 (0 또는 1)
- Y축: 출력 특징의 크기 (L2 Norm)
- 해석: 모델이 오버랩 신호를 얼마나 신뢰하는지 확인


---

##### 2. 모델 평가 함수

<small>**evaluate_pairwise_model: 성능 지표 및 시각화**</small>

**처리 단계**

| 단계 | 설명 |
|------|------|
| 1️⃣ 테스트 데이터 로더 | AugmentedPairDataset + DataLoader |
| 2️⃣ 모델 추론 | batch마다 예측값 수집 |
| 3️⃣ 게이트 값 시각화 | `visualize_gate_values()` 호출 |
| 4️⃣ 성능 리포트 | Classification Report 출력 |
| 5️⃣ 혼동 행렬 | Confusion Matrix 히트맵 |
| 6️⃣ 핵심 지표 | Accuracy & Recall 요약 |

---

**평가 플로우**

<pre style="padding: 12px; border-radius: 5px; overflow-x: auto;">
Test Samples
    │
    ├─ AugmentedPairDataset 생성
    │
    ├─ model.eval() 모드
    │
    ├─ torch.no_grad() 기울기 비활성화
    │
    ├─ Forward Pass
    │  └─ outputs = model(batch_features)
    │
    ├─ 예측값 수집
    │  ├─ all_preds
    │  └─ all_labels
    │
    └─ 평가 지표 계산
       │
       ├─ Classification Report
       │  └─ Precision, Recall, F1-score
       │
       ├─ Confusion Matrix 시각화
       │  └─ TN, FP, FN, TP
       │
       └─ 핵심 지표
          ├─ ✅ Accuracy = (TP+TN) / 전체
          └─ 🎯 Recall (Matching) = TP / (TP+FN)
</pre>


---

##### 3. 혼동 행렬 (Confusion Matrix)

| | **예측: 비매칭** | **예측: 매칭** |
|---|---|---|
| **실제: 비매칭** | TN (True Negative) | FP (False Positive) |
| **실제: 매칭** | FN (False Negative) | TP (True Positive) |

- **TN**: 올바르게 비매칭 예측
- **FP**: 잘못된 매칭 예측
- **FN**: 놓친 매칭 (중요!)
- **TP**: 올바르게 매칭 예측

---

##### 4. 핵심 평가 지표

<small>**Test Tables 기준 평가**</small>

| 지표 | 수식 | 의미 |
|------|------|------|
| **Accuracy** | (TP+TN) / 전체 | 전체 정확도 |
| **Recall** | TP / (TP+FN) | FK 재현율 (매칭 놓침 방지) |
| **Precision** | TP / (TP+FP) | 예측된 매칭의 정확도 |
| **F1-score** | 2 × (Precision × Recall) / (Precision + Recall) | 조화평균 |

**출력 예시**
✅ 최종 테스트 정확도: 94.23%
🎯 Matching(FK) 재현율(Recall): 87.50%

#### 13. 특징 중요도 시각화 (Feature Importance)

##### 1. 특징 중요도 함수

<small>**plot_expert_importance: 172차원 특징 그룹별 중요도 분석**</small>

**입력 데이터 구조**

<pre style="padding: 12px; border-radius: 5px;">
172차원 입력
├─ u (42): [Stat(27) + Char(15)]
├─ v (42): [Stat(27) + Char(15)]
├─ diff (42): [Stat(27) + Char(15)]
├─ mul (42): [Stat(27) + Char(15)]
└─ ov (4): Overlap features
</pre>

**처리 단계**

| 단계 | 설명 |
|------|------|
| 1️⃣ 블록 분리 | u, v, diff, mul, ov로 구분 |
| 2️⃣ 세부 분리 | 각 블록에서 Stat(0:27) & Char(27:42) 추출 |
| 3️⃣ 중요도 계산 | 3개 카테고리별 평균값 계산 |
| 4️⃣ 시각화 | Bar chart로 표현 |

**3개 특징 그룹**

| 그룹 | 포함 범위 | 설명 |
|------|----------|------|
| **Global Stat** | u_s, v_s, d_s, m_s (각 27차원) | 전역 통계 특징 |
| **Char Dist** | u_c, v_c, d_c, m_c (각 15차원) | 문자 분포 특징 |
| **Overlap (Fact)** | ov (4차원) | 오버랩 팩트 특징 |

---

##### 2. 데이터 준비 & 차원 조정

<small>**샘플 데이터 구성 및 차원 정렬**</small>

**데이터 변환 플로우**

<pre style="padding: 12px; border-radius: 5px; overflow-x: auto;">
test_samples_scaled[sample_idx]
    │
    ├─ 원본 구조: [u(42), v(42), ov(5)] = 89차원
    │
    ├─ Step 1: u, v, ov 추출
    │
    ├─ Step 2: diff & mul 수동 생성
    │   ├─ diff = |u - v|  (42차원)
    │   └─ mul = u × v    (42차원)
    │
    ├─ Step 3: 순서대로 결합
    │   └─ [u(42) + v(42) + diff(42) + mul(42) + ov(5)]
    │   └─ = 173차원 (과도)
    │
    ├─ Step 4: 차원 조정 (173 → 172)
    │   ├─ main_feat = flat_sample[:168]
    │   ├─ ov_feat = flat_sample[168:172]  (5번째 값 제거)
    │   └─ 결과: [u + v + diff + mul + ov(4)]
    │
    └─ Step 5: Tensor 변환 & 모델 입력
        └─ sample_input: [1, 172]
</pre>

**차원 정렬 테이블**

| 단계 | 구성 | 차원 | 비고 |
|------|------|------|------|
| 원본 | u, v, ov | 89 | 데이터셋 저장 형식 |
| diff 추가 | u, v, diff, mul, ov | 173 | 모델 입력 구성 |
| **최종** | **u, v, diff, mul, ov(4)** | **172** | ✅ 모델 기대 차원 |

---

##### 3. 실행 로직

<small>**매칭 샘플 선택 및 시각화 실행**</small>

**실행 조건**

<pre style="padding: 12px; border-radius: 5px;">
1. test_labels에서 label=1(Matching) 샘플 찾기
   └─ matching_indices 추출

2. 첫 번째 매칭 샘플 선택
   └─ sample_idx = matching_indices[0]

3. 데이터 처리
   ├─ 1D 배열 변환
   ├─ diff, mul 계산
   ├─ 순서대로 결합
   └─ 차원 조정 (173 → 172)

4. Tensor 변환
   └─ torch.from_numpy() + unsqueeze(0)

5. 시각화 실행
   └─ plot_expert_importance(model, sample_input, device)
</pre>

**출력 메시지**
- ✅ `"🚀 최종 입력 Shape: torch.Size([1, 172])"`
- ⚠️ `"⚠️ 오버랩 차원 조정: 173 -> 172"` (필요시)

---

##### 4. 시각화 결과

**Bar Chart: Feature Group Magnitude**

| 특징 그룹 | 평균값 | 해석 |
|-----------|--------|------|
| 1. Global Stat | - | 수치 통계의 중요도 |
| 2. Char Dist | - | 문자 분포의 중요도 |
| 3. Overlap (Fact) | - | 오버랩 팩트의 중요도 |

(실제 값은 모델이 학습한 샘플마다 다름)


In [ ]:
# [cell13]
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_expert_importance(model, sample_input, device):
    """
    172차원(Stat, Char, Overlap) 전용 중요도 시각화 함수
    """
    # 텐서를 1차원 numpy 배열로 변환
    arr = sample_input.squeeze(0).cpu().numpy()

    # 1. 4개의 블록(u, v, diff, mul)과 1개의 오버랩(ov)으로 분리
    # 각 블록의 크기는 42 (Stat 27 + Char 15) 입니다.
    u = arr[0:42]
    v = arr[42:84]
    diff = arr[84:126]
    mul = arr[126:168]
    ov = arr[168:174] # arr[168:172] # 오버랩은 마지막 4차원

    # 2. 각 블록에서 Stat(0~27)과 Char(27~42) 분리
    u_s, u_c = u[0:27], u[27:42]
    v_s, v_c = v[0:27], v[27:42]
    d_s, d_c = diff[0:27], diff[27:42]
    m_s, m_c = mul[0:27], mul[27:42]

    # 3. 각 카테고리별 평균 계산 (Fingerprint와 BGE는 삭제됨)
    importance_dict = {
        '1. Global Stat': np.mean([u_s.mean(), v_s.mean(), d_s.mean(), m_s.mean()]),
        '2. Char Dist': np.mean([u_c.mean(), v_c.mean(), d_c.mean(), m_c.mean()]),
        '3. Overlap (Fact)': ov.mean()
    }

    # 4. 시각화 (Bar Chart)
    labels = list(importance_dict.keys())
    values = list(importance_dict.values())

    plt.figure(figsize=(8, 5))
    sns.barplot(x=values, y=labels, palette='viridis')
    plt.title('Feature Group Magnitude (Stat, Char, Overlap)')
    plt.xlabel('Mean Absolute Value')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# --- 시각화 실행 부분 (새로운 차원 172 정밀 조정 버전) ---
labels_arr = np.atleast_1d(test_labels)
matching_indices = np.where(labels_arr == 1)[0]

if len(matching_indices) > 0:
    sample_idx = matching_indices[0]
    raw_sample = test_samples_scaled[sample_idx]

    # 1. 모든 요소를 1차원 배열로 변환 후 합치기
    flat_sample = np.concatenate([np.atleast_1d(x) for x in raw_sample[:4]]).astype(np.float32)  # [:4] to exclude pair_key

    # 2. 데이터 구성 확인 및 부족분(diff, mul) 수동 생성
    # 현재 flat_sample은 89차원 [u(42), v(42), ov(5)] 구조입니다.
    if len(flat_sample) < 174: # 172:
        u = flat_sample[0:42]
        v = flat_sample[42:84]
        ov = flat_sample[84:] # 여기서 5차원이 나옴

        diff = np.abs(u - v)
        mul = u * v

        # 모델이 기대하는 순서대로 결합: u, v, diff, mul, ov
        flat_sample = np.concatenate([u, v, diff, mul, ov]).astype(np.float32)

    # 3. 모델의 d5=4 설정과 데이터의 ov=5 차원 충돌 해결
    # 현재 flat_sample은 u(42)+v(42)+diff(42)+mul(42)+ov(6) = 174차원입니다.
    # 모델은 174차원을 기대하므로, 마지막 오버랩의 6번째 값을 버립니다.
    if len(flat_sample) >  174: # 172:
        print(f"⚠️ 오버랩 차원 조정: {len(flat_sample)} -> 174")
        # 앞의 168개(u,v,diff,mul) + 오버랩 앞 5개
        main_feat = flat_sample[:168]
        ov_feat = flat_sample[168:174] # 172] # 뒤에 남는 1개(5번째 값)는 버림
        flat_sample = np.concatenate([main_feat, ov_feat])

    # 4. 텐서 변환 및 실행
    sample_input = torch.from_numpy(flat_sample).unsqueeze(0).to(device)
    print(f"🚀 최종 입력 Shape: {sample_input.shape}") # [1, 172] 여야 함

    plot_expert_importance(model, sample_input, device)

else:
    print("매칭 샘플이 없습니다.")


#### 14. 학습 곡선 시각화 (Learning Curves)

##### 함수: plot_learning_curves

<small>**Train Loss vs Validation Loss 변화 추적**</small>

**목적**
- 학습 진행 과정 시각화
- 과적합(Overfitting) 발생 시점 파악
- 최적 모델이 저장된 epoch 확인

---

**차트 구성 요소**

| 요소 | 설명 | 색상 | 스타일 |
|------|------|------|--------|
| **Train Loss** | 훈련 데이터 손실 | 파란색 (#1f77b4) | 점선(--) |
| **Val Loss** | 검증 데이터 손실 | 주황색 (#ff7f0e) | 실선 |
| **Best Model** | Val Loss 최소 지점 | 빨간색 | 점선(:) |

---

**시각화 로직**

<pre style="padding: 12px; border-radius: 5px; overflow-x: auto;">
Loss 데이터 입력
    │
    ├─ 유효성 검사
    │  └─ train_losses, val_losses 존재 확인
    │
    ├─ 그래프 생성
    │  └─ Figure(10×5)
    │
    ├─ 1. Loss Curve 그리기
    │  ├─ plt.plot(train_losses) ─ 파란 점선
    │  └─ plt.plot(val_losses) ──── 주황 실선
    │
    ├─ 2. 최적 모델 시점 표시
    │  ├─ min_val_idx = argmin(val_losses)
    │  └─ plt.axvline(x=min_val_idx) ─ 빨간 수직선
    │
    ├─ 3. 범례 & 그리드 추가
    │  ├─ legend(upper right)
    │  └─ grid(True, alpha=0.6)
    │
    └─ 4. 출력
       └─ plt.show()
</pre>

---

**해석 가이드**

**정상적인 학습 곡선**
Loss
│      Train ····
│       ╱╲
│      ╱  ╲      Validation ────
│     ╱    ╲╱╲
│    ╱        ╲
└────────────────── Epochs
↑
Best Model


- Train Loss: 점진적 감소
- Val Loss: 감소 후 안정화 또는 증가
- **최적 모델**: Val Loss 최소 지점

**과적합 신호**
- Train Loss는 계속 감소
- Val Loss는 증가 추세
- 두 곡선 사이 간격 벌어짐

---

**실행**

```python
# 학습 루프에서 생성된 리스트로 시각화
plot_learning_curves(train_losses_list, val_losses_list)

출력

📈 제목: "Pairwise Matching Model: Training & Validation Loss"
📊 X축: Epochs (에포크 번호)
📉 Y축: Loss (Focal Loss 값)
🎯 마크: Best Model 지점 (수직선)

In [ ]:
# [cell14]
import matplotlib.pyplot as plt

def plot_learning_curves(train_losses, val_losses):
    """
    학습 과정에서의 Train Loss와 Validation Loss 변화를 시각화합니다.
    과적합(Overfitting) 발생 시점을 파악하기 용이합니다.
    """
    if not train_losses or not val_losses:
        print("💡 시각화할 Loss 기록이 부족합니다. 학습 루프에서 losses 리스트가 채워졌는지 확인하세요.")
        return

    plt.figure(figsize=(10, 5))

    # 1. Loss Curve
    plt.plot(train_losses, label='Train Loss', color='#1f77b4', linewidth=2, linestyle='--')
    plt.plot(val_losses, label='Validation Loss', color='#ff7f0e', linewidth=2)

    # 최적 모델 시점 표시 (Val Loss가 최소인 지점)
    min_val_idx = val_losses.index(min(val_losses))
    plt.axvline(x=min_val_idx, color='red', linestyle=':', label=f'Best Model (Epoch {min_val_idx+1})')

    plt.title('Pairwise Matching Model: Training & Validation Loss', fontsize=14, pad=15)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss (Focal Loss)', fontsize=12)
    plt.legend(loc='upper right', frameon=True)
    plt.grid(True, linestyle='--', alpha=0.6)

    # 여백 조정 및 출력
    plt.tight_layout()
    plt.show()

# 학습 루프에서 생성된 리스트로 시각화 실행
# plot_learning_curves(train_losses_list, val_losses_list)


#### 17. 폴더 기반 FK 추론 및 성능 평가

##### 폴더 내 모든 CSV 파일을 자동 탐색하고, ground_truth.json 기준으로 성능 지표 계산

**특징:**
- 폴더 내 모든 CSV 쌍 × 컬럼 쌍 자동 조합
- 컬럼별 피처(TaskA+B) 사전 계산으로 중복 방지
- Jaccard/Containment 기반 하드 제약(overlap=0이면 score=0)
- Precision, Recall, F1 자동 계산

In [ ]:
# [cell 17] 폴더 기반 FK 추론
import itertools
import pathlib
import json as _json
from tqdm.auto import tqdm

def infer_folder(
    folder_path,
    model,
    extractor,
    scaler,
    ov_scaler,
    device=None,
    threshold=0.5,
    apply_hard_constraint=True,
    verbose=True,
):
    """폴더 내 CSV 파일 전체에 대해 FK 추론 및 성능 평가
    
    Args:
        folder_path: CSV 파일과 ground_truth.json이 있는 폴더
        model: 학습된 CRIM_Statisticsnly_Matcher 모델
        extractor: FeatureExtractor 인스턴스
        scaler: 컬럼 피처용 StandardScaler
        ov_scaler: overlap 피처용 StandardScaler
        device: torch device
        threshold: FK 판정 스코어 threshold (기본 0.5)
        apply_hard_constraint: raw overlap=0이면 score=0 적용 여부
        verbose: 진행상황 출력 여부
    
    Returns:
        dict: 추론 결과 및 성능 지표
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    folder = pathlib.Path(folder_path)
    model.eval()
    
    # Step 1: CSV 파일 탐색
    all_csv_files = sorted(folder.rglob('*.csv'))
    csv_files = [
        path for path in all_csv_files
        if not path.name.startswith('spd_') and 'spider' not in path.name.lower()
    ]
    csv_pairs = list(itertools.combinations(csv_files, 2))
    if verbose:
        print(f'✓ 폴더 스캔 완료: {len(csv_files)}개 CSV, {len(csv_pairs)}개 쌍')
        excluded_count = len(all_csv_files) - len(csv_files)
        if excluded_count:
            print(f'  spd_/spider CSV 제외: {excluded_count}개')
    
    # Step 2: Ground Truth 로드
    gt_path = folder / 'ground_truth.json'
    with open(gt_path) as f:
        gt_list = _json.load(f)  # [["table.col", "table.col"], ...]
    
    # GT canonical 집합 (방향 무관 비교)
    gt_canonical = set()
    for item in gt_list:
        ta, ca = item[0].split('.')
        tb, cb = item[1].split('.')
        gt_canonical.add(frozenset({(ta, ca), (tb, cb)}))
    if verbose:
        print(f'✓ Ground Truth 로드: {len(gt_canonical)}개 FK 쌍')
        print(gt_canonical)
    
    # Step 3: CSV 로드 및 컬럼 피처 사전 계산
    col_features = {}  # {(csv_path_str, col_name): np.array (42-dim raw)}
    loaded_dfs = {}    # {csv_path_str: DataFrame}
    
    for csv_path in tqdm(csv_files, desc="load CSVs"):
        try:
            df = pd.read_csv(
                csv_path, encoding='utf-8-sig',
                on_bad_lines='skip', low_memory=False
            )  # v13 patch: nrows=10000 제거 → 학습/추론 분포 일치
            loaded_dfs[str(csv_path)] = df
            
            for col in df.columns:
                feat = extractor.task4_combine_features(df[col])
                col_features[(str(csv_path), col)] = np.array(feat).flatten()
        except Exception as e:
            if verbose:
                print(f'  ⚠ {csv_path.stem}: {e}')
    
    if verbose:
        print(f'✓ 컬럼 피처 추출 완료')
    
    # Step 4: 배치 추론
    all_results = []
    pair_count = 0
    
    with torch.no_grad():
        for path_a, path_b in tqdm(csv_pairs, desc="infer pairs"):
            name_a, name_b = path_a.stem, path_b.stem
            
            if str(path_a) not in loaded_dfs or str(path_b) not in loaded_dfs:
                continue
            
            df_a = loaded_dfs[str(path_a)]
            df_b = loaded_dfs[str(path_b)]
            
            for col_a in df_a.columns:
                for col_b in df_b.columns:
                    pair_count += 1
                    
                    f1_raw = col_features.get((str(path_a), col_a))
                    f2_raw = col_features.get((str(path_b), col_b))
                    
                    if f1_raw is None or f2_raw is None:
                        continue
                    
                    # 스케일링
                    f1_s = scaler.transform(f1_raw.reshape(1, -1)).flatten()
                    f2_s = scaler.transform(f2_raw.reshape(1, -1)).flatten()
                    
                    # Overlap 피처 계산 (raw → scale 적용 전 값으로 하드 제약 판단)
                    raw_ov = compute_overlap_features(df_a[col_a], df_b[col_b])
                    ov_scaled = ov_scaler.transform(
                        np.array(raw_ov).reshape(1, -1)
                    ).flatten().tolist()
                    
                    # v30 layout: [u_stat(18), u_char(18), v_stat(18), v_char(18), pair_stat(6), ov(20)] = 98-dim
                    # pair_stat 6 derive (training Dataset과 동일 로직)
                    EPS = 1e-6
                    pair_stat = np.array([
                        abs(f1_s[2] - f2_s[2]),                       # ratio_diff
                        f1_s[2] - f2_s[2],                            # unique_ratio_diff (signed)
                        f1_s[1] - f2_s[1],                            # null_ratio_diff (signed)
                        np.tanh(f1_s[7] / (abs(f2_s[7]) + EPS) - 1.0),# range_ratio
                        np.tanh(f1_s[8] / (abs(f2_s[8]) + EPS) - 1.0),# std_ratio
                        np.tanh(f1_s[11]/ (abs(f2_s[11])+ EPS) - 1.0),# entropy_ratio
                    ], dtype=np.float32)
                    u_stat, u_char = f1_s[:18], f1_s[18:36]
                    v_stat, v_char = f2_s[:18], f2_s[18:36]
                    combined = np.concatenate([u_stat, u_char, v_stat, v_char, pair_stat, ov_scaled])
                    combined = np.nan_to_num(combined, nan=0.0, posinf=1.0, neginf=0.0)
                    
                    # 모델 추론
                    input_tensor = torch.FloatTensor(combined).unsqueeze(0).to(device)
                    outputs = model(input_tensor)   # v13: [B] logit
                    # BCE + sigmoid → P(FK)
                    if outputs.ndim == 0:
                        score = torch.sigmoid(outputs).item()
                    else:
                        score = torch.sigmoid(outputs[0]).item()
                    
                    # Hard Constraint: raw overlap 모두 0이면 score=0
                    jaccard = round(raw_ov[0], 4)
                    con_a   = round(raw_ov[1], 4)
                    con_b   = round(raw_ov[2], 4)
                    # if apply_hard_constraint and jaccard == 0 and con_a == 0 and con_b == 0:
                    #     score = 0.0
                    
                    all_results.append({
                        'A_table': name_a, 'A_col': col_a,
                        'B_table': name_b, 'B_col': col_b,
                        'FK_Score': round(score, 4),
                        # 'Non_FK_Score': round(score_n, 4),
                        'Jaccard': jaccard,
                        'Con_A': con_a,
                        'Con_B': con_b,
                    })
    
    if verbose:
        print(f'✓ 배치 추론 완료: {pair_count}개 컬럼 쌍 검사')
    
    # Step 5: 성능 지표 계산
    predicted = [r for r in all_results if r['FK_Score'] >= threshold]
    
    pred_canonical = set(
        frozenset({(r['A_table'], r['A_col']), (r['B_table'], r['B_col'])})
        for r in predicted
    )
    
    tp = len(gt_canonical & pred_canonical)
    fp = len(pred_canonical) - tp
    fn = len(gt_canonical) - tp
    tn = pair_count - tp - fp - fn
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    result = {
        'folder': str(folder),
        'n_csv': len(csv_files),
        'n_pairs_checked': pair_count,
        'threshold': threshold,
        'metrics': {
            'precision': round(precision, 4),
            'recall':    round(recall, 4),
            'f1':        round(f1, 4),
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        },
        'predicted':   predicted,
        'all_results': all_results,
    }
    
    if verbose:
        print(f'\n[성능 지표]')
        print(f'  전체 컬럼 쌍: {pair_count}개')
        print(f'  Threshold : {threshold}')
        print(f'  TP={tp}, FP={fp}, FN={fn}, TN={tn}')
        print(f'  Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}')
    
    return result


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 📍 추론 DB 단일 설정 — 이 셀에서만 정의, 다운스트림 모두 참조
# ────────────────────────────────────────────────────────────────────
# 두 가지 방식 중 선택:
#   (A) USE_MASKED 토글 — 간단, 자동으로 _mask 접미사 처리
#   (B) INFERENCE_DBS_OVERRIDE 직접 지정 — 유연, 8개 혼합/일부 선택 등
# ────────────────────────────────────────────────────────────────────

# ★ 평가 대상 DB 베이스 이름 (마스킹 접미사 없는 원형) ★
_BIRD_BASE_DBS = [
    'toxicology',
    'california_schools',
    'superhero',
    'thrombosis_prediction',
]

# ── (A) USE_MASKED 토글 (override 없을 때만 사용) ─────────────────
USE_MASKED = False    # True → *_mask 4개, False → 원본 4개

# ── (B) 직접 지정 (override) — None 이면 USE_MASKED 사용 ─────────
# 사용 예: 8개 한 번에 비교, 특정 DB 만 보기
INFERENCE_DBS_OVERRIDE = None
# INFERENCE_DBS_OVERRIDE = _BIRD_BASE_DBS + [f'{db}_mask' for db in _BIRD_BASE_DBS]  # 8개 한꺼번에
# INFERENCE_DBS_OVERRIDE = ['superhero', 'superhero_mask']                            # 페어 비교
# INFERENCE_DBS_OVERRIDE = ['superhero']                                              # 단일
# INFERENCE_DBS_OVERRIDE = ['california_schools_mask', 'toxicology_mask', 'superhero_mask', 'thrombosis_prediction_mask']

# ── 자동 파생 ────────────────────────────────────────────────────
TEST_ROOT = '../workspace/dataset_mini/bird_data_tables'

if INFERENCE_DBS_OVERRIDE is not None:
    INFERENCE_DBS = list(INFERENCE_DBS_OVERRIDE)
    # 파일명 충돌 방지를 위한 자동 suffix
    has_orig = any(not db.endswith('_mask') for db in INFERENCE_DBS)
    has_mask = any(db.endswith('_mask')     for db in INFERENCE_DBS)
    if has_orig and has_mask:
        RUN_SUFFIX = '_all'         # 혼합 → 별도 파일
    elif has_mask:
        RUN_SUFFIX = '_mask'
    else:
        RUN_SUFFIX = ''
else:
    INFERENCE_DBS = (
        [f'{db}_mask' for db in _BIRD_BASE_DBS] if USE_MASKED else list(_BIRD_BASE_DBS)
    )
    RUN_SUFFIX = '_mask' if USE_MASKED else ''

SINGLE_INFER_DB = INFERENCE_DBS[0]
TEST_DBS        = INFERENCE_DBS
test_root       = TEST_ROOT

print(f"📍 추론 DB 설정")
print(f"  모드             : {'override' if INFERENCE_DBS_OVERRIDE is not None else f'USE_MASKED={USE_MASKED}'}")
print(f"  INFERENCE_DBS    : {INFERENCE_DBS}")
print(f"  SINGLE_INFER_DB  : {SINGLE_INFER_DB}")
print(f"  TEST_ROOT        : {TEST_ROOT}")
print(f"  RUN_SUFFIX       : '{RUN_SUFFIX}' (저장 파일명에 자동 적용)")


In [ ]:
# [v34_final_ablation] variant 1개 선택 추론
# ── 안전장치: post_fix 검증 (다른 노트북의 post_fix 로 v34 파일 덮어쓰기 방지) ──
assert 'post_fix' in dir(), 'cell 4 (post_fix 설정) 를 먼저 실행하세요.'
assert post_fix == '0517_v34_final_ablation', (
    f'⛔ post_fix="{post_fix}" — v34 원본 노트북의 기대값과 다름. '
    f'cell 4 를 다시 실행하세요. (예상값: 0517_v34_final_ablation)'
)
print(f'  ✓ post_fix={post_fix} 검증 통과')

import pandas as pd
import torch
import numpy as np
import joblib
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
extractor = FeatureExtractor(dataset_path)

ablation_tag = 'stat_char_ov'
ablation_model_path = os.path.join(
    base_path, f'multi_task_matching_model_{post_fix}_{ablation_tag}.pth'
)

print(f"🔄 모델 로드 (variant: {ablation_tag}, config: α={FOCAL_ALPHA}, γ={FOCAL_GAMMA})")
checkpoint = torch.load(ablation_model_path, map_location=device)
scaler = joblib.load(scaler_path)

branch_config = checkpoint.get('branch_config', ['stat', 'char', 'ov'])
s_dim         = checkpoint.get('s_dim', 36)
loaded_seed   = checkpoint.get('seed', 'unknown')

model = V33AblationMatcher(branch_config=branch_config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Loaded: {os.path.basename(ablation_model_path)}  branch_config={branch_config}  seed={loaded_seed}")

ov_scaler = joblib.load(ov_scaler_path)
clip_lower, clip_upper = joblib.load(clip_bounds_path)

db_name = SINGLE_INFER_DB
test_folder = f'{TEST_ROOT}/{db_name}'

result = infer_folder(
    test_folder, model, extractor, scaler, ov_scaler,
    device=device, threshold=0.5, apply_hard_constraint=True, verbose=True,
)

print(f'\n결과 요약 (seed={loaded_seed}):')
print(f'  발견된 FK 후보: {len(result["predicted"])}개')
print(f'  메트릭: {result["metrics"]}')


In [ ]:
# [cell 19] 추론 결과 분석
if result and result['predicted']:
    # GT canonical 집합 재구성 (infer_folder 내부와 동일)
    import json as _json2, pathlib as _pathlib
    _gt_path = _pathlib.Path(result['folder']) / 'ground_truth.json'
    with open(_gt_path) as _f:
        _gt_list = _json2.load(_f)
    _gt_canonical = set()
    for _item in _gt_list:
        _ta, _ca = _item[0].split('.')
        _tb, _cb = _item[1].split('.')
        _gt_canonical.add(frozenset({(_ta, _ca), (_tb, _cb)}))

    # 상위 결과 표시 + TP 여부 컬럼 추가
    df_pred = pd.DataFrame(result['predicted']).sort_values('FK_Score', ascending=False)
    df_pred['TP'] = df_pred.apply(
        lambda r: '✓' if frozenset({(r['A_table'], r['A_col']), (r['B_table'], r['B_col'])}) in _gt_canonical else '',
        axis=1
    )
    display(df_pred.head(100))

    # 스코어 분포
    df_all = pd.DataFrame(result['all_results'])
    print(f'\n스코어 분포:')
    print(f'  >= 0.9: {(df_all["FK_Score"] >= 0.9).sum()}개')
    print(f'  >= 0.7: {(df_all["FK_Score"] >= 0.7).sum()}개')
    print(f'  >= 0.5: {(df_all["FK_Score"] >= 0.5).sum()}개')
    print(f'  >= 0.3: {(df_all["FK_Score"] >= 0.3).sum()}개')


In [ ]:
# [v34_final_ablation] 7개 모델 × 4 BIRD DB 추론 + 비교 표 + JSON 저장 (이후 multi-seed 집계용)
# ★ 재실행 정책: 셀 실행 시 항상 모든 (config × db) 재계산하여 JSON 덮어쓰기 ★
# ── 안전장치: post_fix 검증 (다른 노트북의 post_fix 로 v34 파일 덮어쓰기 방지) ──
assert 'post_fix' in dir(), 'cell 4 (post_fix 설정) 를 먼저 실행하세요.'
assert post_fix == '0517_v34_final_ablation', (
    f'⛔ post_fix="{post_fix}" — v34 원본 노트북의 기대값과 다름. '
    f'cell 4 를 다시 실행하세요. (예상값: 0517_v34_final_ablation)'
)
print(f'  ✓ post_fix={post_fix} 검증 통과')

import os
import json
import torch
import pathlib

# ★ TEST_DBS / test_root 는 위 DB config 셀에서 정의됨 ★
# (USE_MASKED 토글로 마스킹/비마스킹 전환)
# 여기서는 그대로 참조만 — 별도 정의 안 함
assert 'TEST_DBS' in dir() and 'test_root' in dir(), \
    "❗ DB config 셀을 먼저 실행하세요 (cell 40)"

results_table = {}

for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    save_path = ablation_paths[tag]
    ckpt = torch.load(save_path, map_location=device)
    # ★ 모델 로드 시점에도 seed 리셋 (학습 때와 동일 RNG 상태 보장) ★
    apply_seed(SEED)
    model = V33AblationMatcher(branch_config=cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    results_table[tag] = {}
    for db_name in TEST_DBS:
        folder = pathlib.Path(test_root) / db_name
        # ★ 추론 직전 RNG 리셋 — 같은 데이터·같은 가중치면 같은 score 보장 ★
        # (이 보정 없으면 RNG 상태가 이전 (config, db) 추론에 의해 변경되어
        #  같은 logical 데이터인 superhero vs superhero_mask 가 다른 sampling
        #  → 다른 features → 다른 borderline FP 를 산출함)
        apply_seed(SEED)
        try:
            result = infer_folder(
                folder_path=str(folder), model=model, extractor=extractor,
                scaler=scaler, ov_scaler=ov_scaler, device=device,
                threshold=0.5, apply_hard_constraint=True, verbose=False,
            )
            m = result.get('metrics', {})
        except Exception as e:
            print(f'  ⚠ {tag} | {db_name}: {e}')
            m = {'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'tp': 0, 'fp': 0, 'fn': 0}
        results_table[tag][db_name] = m
    clear_memory()

# ── 출력 ──
print('\n' + '=' * 100)
print(f'=== v34_final_ablation 7 variants 상세 결과 (SEED={SEED}) ===')
print('=' * 100)
hdr = f'{"config":<18} | {"DB":<22} | {"F1":>6} | {"Prec":>6} | {"Recall":>6} | {"TP":>3} {"FP":>4} {"FN":>3}'
print(hdr); print('-' * 100)
for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    for db_name in TEST_DBS:
        m = results_table[tag][db_name]
        print(f'{tag:<18} | {db_name:<22} | {m["f1"]:>6.4f} | {m["precision"]:>6.4f} | {m["recall"]:>6.4f} | {m["tp"]:>3} {m["fp"]:>4} {m["fn"]:>3}')
    print('-' * 100)

print('\n' + '=' * 80)
print(f'=== F1 요약 (config × DB, SEED={SEED}) ===')
print('=' * 80)
print(f'{"config":<18} | ' + ' | '.join(f'{db[:14]:>14}' for db in TEST_DBS))
print('-' * 80)
for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    row = f'{tag:<18} | ' + ' | '.join(f'{results_table[tag][db]["f1"]:>14.4f}' for db in TEST_DBS)
    note = '  ← baseline (v33 full)' if cfg == ['stat', 'char', 'ov'] else ''
    print(row + note)

print('\n' + '=' * 60)
print(f'=== 평균 F1 순위 (4 DB 평균, SEED={SEED}) ===')
print('=' * 60)
avg_scores = []
for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    avg_f1 = sum(results_table[tag][db]['f1'] for db in TEST_DBS) / len(TEST_DBS)
    avg_scores.append((avg_f1, tag))
avg_scores.sort(reverse=True)
for i, (avg_f1, tag) in enumerate(avg_scores, 1):
    print(f'  {i}. {tag:<20} avg F1 = {avg_f1:.4f}')

# ── JSON 저장 (multi-seed 집계용) ──
results_json_path = os.path.join(base_path, f'ablation_results_{post_fix}{RUN_SUFFIX}.json')
# 마스킹/비마스킹은 서로 다른 파일로 분리 저장 (RUN_SUFFIX 자동 적용)
with open(results_json_path, 'w') as f:
    json.dump({
        'seed': SEED,
        'focal_alpha': FOCAL_ALPHA,
        'focal_gamma': FOCAL_GAMMA,
        'results_table': results_table,
        'avg_scores': avg_scores,
    }, f, indent=2)
print(f'\n💾 저장: {results_json_path}')
print('  → Cell 43 (마지막)에서 baseline 비교')


In [ ]:
# [v34_final_ablation] 결과 요약 — baseline 비교
import os
import json

results_json_path = os.path.join(base_path, f'ablation_results_{post_fix}{RUN_SUFFIX}.json')
if os.path.exists(results_json_path):
    with open(results_json_path) as f:
        data = json.load(f)
    print('=== v34_final_ablation 결과 요약 ===')
    print(f'  SEED: {data.get("seed")}, FocalLoss(α={data.get("focal_alpha")}, γ={data.get("focal_gamma")})')
    print()
    rt = data['results_table']
    CONFIGS = ['stat', 'char', 'ov', 'stat+char', 'stat+ov', 'char+ov', 'stat+char+ov']
    # ★ DBs 는 위 DB config 의 INFERENCE_DBS / TEST_DBS 재사용 ★
    DBs = TEST_DBS
    print(f'{"config":<18} | ' + ' | '.join(f'{db[:14]:>14}' for db in DBs) + ' |   avg')
    print('-' * 100)
    for cfg in CONFIGS:
        row = f'{cfg:<18} | ' + ' | '.join(f'{rt[cfg][db]["f1"]:>14.4f}' for db in DBs)
        avg = sum(rt[cfg][db]['f1'] for db in DBs) / len(DBs)
        note = '  ← v34 baseline (full)' if cfg == 'stat+char+ov' else ''
        print(row + f' | {avg:>6.4f}' + note)

    print()
    print('=== 참고 baseline 비교 ===')
    print('  v33_trap5 (BCE, seed=42):                  super=0.357, avg=0.706')
    print('  v34_final (Focal α=0.5 γ=2, seed=42):      super≈0.488, avg≈0.739')
    print('  → v34_final_ablation은 동일 loss로 7 변종 비교')
else:
    print(f'⚠ 결과 JSON 없음 ({results_json_path})')
    print('  → Cell 42 실행 후 다시 이 셀 실행')


In [ ]:
# ── PR-AUC + Recall@K — 모든 ablation config × DB ──────────────────
# Valentine / EmbDI / DeepJoin / 통합 비교 노트북과 같은 스키마로 저장:
#   /source/ablation_ranking_metrics_{post_fix}.json
#
# 28회 추론 (7 configs × 4 DBs) 이 비싸기 때문에:
#   1) 기존 JSON 이 있으면 로드하여 이미 완료된 (config, db) 는 건너뜀
#   2) 매 (config, db) 완료 시점마다 JSON 즉시 저장 → 중단되어도 진척 보존
#   3) 같은 (config, db) 를 다시 돌리고 싶으면 FORCE_RECOMPUTE=True

# ── 안전장치: post_fix 검증 (다른 노트북의 post_fix 로 v34 파일 덮어쓰기 방지) ──
assert 'post_fix' in dir(), 'cell 4 (post_fix 설정) 를 먼저 실행하세요.'
assert post_fix == '0517_v34_final_ablation', (
    f'⛔ post_fix="{post_fix}" — v34 원본 노트북의 기대값과 다름. '
    f'cell 4 를 다시 실행하세요. (예상값: 0517_v34_final_ablation)'
)
print(f'  ✓ post_fix={post_fix} 검증 통과')

from sklearn.metrics import average_precision_score
import numpy as np
import json
import pathlib
import os

# ★ 재실행 정책 ★
# True (기본) : 셀 재실행하면 모든 (config, db) 항상 다시 계산 + JSON 값 업데이트
# False        : 이미 결과 있는 (config, db) 는 스킵 (resume 모드 — 중단 후 재개용)
FORCE_RECOMPUTE = True
RANKING_JSON_PATH = os.path.join(
    base_path, f'ablation_ranking_metrics_{post_fix}{RUN_SUFFIX}.json'
)

def compute_ranking_metrics(all_results, gt_canonical):
    """CRIM의 all_results와 GT로부터 PR-AUC, Recall@K 계산."""
    K = len(gt_canonical)
    total = len(all_results)

    y_true = np.array([
        1 if frozenset({(r['A_table'], r['A_col']),
                        (r['B_table'], r['B_col'])}) in gt_canonical else 0
        for r in all_results
    ])
    y_score = np.array([r['FK_Score'] for r in all_results])

    pr_auc = average_precision_score(y_true, y_score) if y_true.sum() > 0 else float('nan')

    # MRR — 각 true positive 의 rank reciprocal 평균
    if y_true.sum() > 0:
        order = np.argsort(-y_score, kind='stable')
        ranked_truth = y_true[order]
        rrs = [1.0 / (i + 1) for i, v in enumerate(ranked_truth) if v == 1]
        mrr = float(np.mean(rrs)) if rrs else float('nan')
    else:
        mrr = float('nan')

    if K > 0 and K <= total:
        top_k_idx = np.argsort(-y_score, kind='stable')[:K]
        tp_at_K = int(y_true[top_k_idx].sum())
        recall_at_K = tp_at_K / K
    else:
        tp_at_K, recall_at_K = 0, float('nan')

    if K > 0 and 3*K <= total:
        top_3k_idx = np.argsort(-y_score, kind='stable')[:3*K]
        recall_at_3K = int(y_true[top_3k_idx].sum()) / K
    else:
        recall_at_3K = float('nan')

    return {
        'pr_auc':       pr_auc,
        'mrr':          mrr,
        'recall_at_K':  recall_at_K,
        'recall_at_3K': recall_at_3K,
        'tp_at_K':      tp_at_K,
        'n_gt':         K,
        'total_pairs':  total,
        'y_true':       y_true.tolist(),
        'y_score':      [round(float(s), 6) for s in y_score],
    }


def load_gt_canonical(folder_path):
    """폴더의 ground_truth.json 을 frozenset 집합으로 변환."""
    with open(pathlib.Path(folder_path) / 'ground_truth.json') as f:
        gt_list = json.load(f)
    gt_canonical = set()
    for item in gt_list:
        ta, ca = item[0].split('.')
        tb, cb = item[1].split('.')
        gt_canonical.add(frozenset({(ta, ca), (tb, cb)}))
    return gt_canonical


def save_progress(ranking_table_disk, seed):
    """매 (config, db) 후 호출 — 진척 즉시 저장."""
    # macro / 순위 계산은 누적 데이터로 재계산
    valid_tags = list(ranking_table_disk.keys())
    avg_summary = []
    for tag in valid_tags:
        dbs = ranking_table_disk[tag]
        pr_vals  = [v['pr_auc']      for v in dbs.values()
                    if v.get('pr_auc') is not None
                    and not (isinstance(v['pr_auc'], float) and np.isnan(v['pr_auc']))]
        mrr_vals = [v['mrr']         for v in dbs.values()
                    if v.get('mrr') is not None
                    and not (isinstance(v['mrr'], float) and np.isnan(v['mrr']))]
        rk_vals  = [v['recall_at_K'] for v in dbs.values()
                    if v.get('recall_at_K') is not None
                    and not (isinstance(v['recall_at_K'], float) and np.isnan(v['recall_at_K']))]
        r3k_vals = [v['recall_at_3K'] for v in dbs.values()
                    if v.get('recall_at_3K') is not None
                    and not (isinstance(v['recall_at_3K'], float) and np.isnan(v['recall_at_3K']))]
        avg_summary.append({
            'tag': tag,
            'avg_pr_auc':       (round(sum(pr_vals)/len(pr_vals), 4)   if pr_vals  else None),
            'avg_mrr':          (round(sum(mrr_vals)/len(mrr_vals), 4) if mrr_vals else None),
            'avg_recall_at_K':  (round(sum(rk_vals)/len(rk_vals), 4)   if rk_vals  else None),
            'avg_recall_at_3K': (round(sum(r3k_vals)/len(r3k_vals), 4) if r3k_vals else None),
            'n_dbs_completed':  len(pr_vals),
        })

    with open(RANKING_JSON_PATH, 'w') as f:
        json.dump({
            'seed': seed,
            'ranking_table': ranking_table_disk,
            'avg_summary':   avg_summary,
        }, f, indent=2)


# ── 1) 기존 진척 로드 ───────────────────────────────────────────
if os.path.exists(RANKING_JSON_PATH) and not FORCE_RECOMPUTE:
    with open(RANKING_JSON_PATH) as f:
        existing = json.load(f)
    ranking_table_disk = existing.get('ranking_table', {})

    # ★ stale DB 제거 — 현재 TEST_DBS 에 없는 DB 는 폐기 ★
    # 이전 USE_MASKED 모드에서 들어온 잔여 데이터가 섞이는 것을 방지.
    # (RUN_SUFFIX 분리 후에는 발생하지 않지만, 과거 누적된 파일 안전장치)
    _stale_removed = 0
    for _tag in list(ranking_table_disk.keys()):
        for _db in list(ranking_table_disk[_tag].keys()):
            if _db not in TEST_DBS:
                del ranking_table_disk[_tag][_db]
                _stale_removed += 1
    if _stale_removed > 0:
        print(f'🧹 stale DB 엔트리 {_stale_removed}개 제거 '
              f'(현재 TEST_DBS={TEST_DBS} 와 무관한 데이터)')

    n_done = sum(len(v) for v in ranking_table_disk.values())
    print(f'📋 기존 결과 로드: {RANKING_JSON_PATH}')
    print(f'    이미 완료된 (config × db) 쌍 수: {n_done}')
else:
    ranking_table_disk = {}
    print(f'📋 신규 실행 — {RANKING_JSON_PATH}')

# ── 2) (config × db) 루프 ─────────────────────────────────────
total_pairs = len(ABLATION_CONFIGS) * len(TEST_DBS)
done_count = 0

for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    if tag not in ranking_table_disk:
        ranking_table_disk[tag] = {}

    # 이 tag 의 모든 DB 가 이미 완료됐으면 모델 로드도 건너뜀
    pending_dbs = [db for db in TEST_DBS
                   if db not in ranking_table_disk[tag] or FORCE_RECOMPUTE]
    if not pending_dbs:
        print(f'⏭  [{tag}] 모든 DB 완료 — 스킵')
        done_count += len(TEST_DBS)
        continue

    # 모델 로드 (필요한 DB 가 하나라도 있을 때만)
    save_path = ablation_paths[tag]
    ckpt = torch.load(save_path, map_location=device)
    # ★ 모델 로드 시점에도 seed 리셋 (학습 때와 동일 RNG 상태 보장) ★
    apply_seed(SEED)
    model = V33AblationMatcher(branch_config=cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    for db_name in TEST_DBS:
        done_count += 1
        if db_name in ranking_table_disk[tag] and not FORCE_RECOMPUTE:
            print(f'  ⏭  [{done_count}/{total_pairs}] {tag} | {db_name} 이미 완료 (resume 모드)')
            continue
        # FORCE_RECOMPUTE=True 면 기존 entry 가 있어도 새로 계산해 덮어씀

        print(f'  ▶  [{done_count}/{total_pairs}] {tag} | {db_name} 추론 중...')
        folder = pathlib.Path(test_root) / db_name
        # ★ 추론 직전 RNG 리셋 (cell 43 와 동일 이유) ★
        apply_seed(SEED)
        try:
            result = infer_folder(
                folder_path=str(folder),
                model=model, extractor=extractor,
                scaler=scaler, ov_scaler=ov_scaler, device=device,
                threshold=0.5, apply_hard_constraint=True, verbose=False,
            )
            gt_canonical = load_gt_canonical(result['folder'])
            metrics = compute_ranking_metrics(result['all_results'], gt_canonical)
        except Exception as e:
            print(f'      ⚠ 실패: {e}')
            metrics = {
                'pr_auc':       None, 'mrr':          None,
                'recall_at_K':  None, 'recall_at_3K': None,
                'tp_at_K':      0,    'n_gt':         0,
                'total_pairs':  0,
            }

        # NaN -> None (JSON 직렬화 안전)
        clean = {k: (None if (isinstance(v, float) and np.isnan(v)) else
                     (round(v, 4) if isinstance(v, float) else v))
                 for k, v in metrics.items()}
        # y_true/y_score 는 list 형 그대로 (이미 위에서 round 처리됨)
        ranking_table_disk[tag][db_name] = clean

        # ★ 매 (config, db) 후 즉시 저장 (중단 안전)
        save_progress(ranking_table_disk, SEED)

        r3k = (f"{clean['recall_at_3K']:.4f}"
               if clean['recall_at_3K'] is not None else '   N/A')
        pr  = (f"{clean['pr_auc']:.4f}"
               if clean['pr_auc'] is not None else '   N/A')
        mrr = (f"{clean['mrr']:.4f}"
               if clean.get('mrr') is not None else '   N/A')
        rk  = (f"{clean['recall_at_K']:.4f}"
               if clean['recall_at_K'] is not None else '   N/A')
        print(f'      → PR-AUC={pr}, MRR={mrr}, R@K={rk}, R@3K={r3k}, |GT|={clean["n_gt"]}')

    clear_memory()

# ── 3) 최종 요약 출력 ───────────────────────────────────────────
print('\n' + '=' * 110)
print(f'=== PR-AUC / Recall@K — v34_final_ablation (SEED={SEED}) ===')
print('=' * 110)
hdr = (f'{"config":<18} | {"DB":<22} | {"PR-AUC":>8} | {"MRR":>8} | '
       f'{"R@K":>8} | {"R@3K":>8} | {"|GT|":>4} | {"#pairs":>7}')
print(hdr); print('-' * 120)
for cfg in ABLATION_CONFIGS:
    tag = '+'.join(cfg)
    for db_name in TEST_DBS:
        m = ranking_table_disk.get(tag, {}).get(db_name)
        if m is None:
            print(f'{tag:<18} | {db_name:<22} | {"  N/A":>8} | {"  N/A":>8} | {"  N/A":>8} | '
                  f'{"  N/A":>8} | {"  -":>4} | {"  -":>7}')
            continue
        pr  = (f"{m['pr_auc']:.4f}"       if m['pr_auc']       is not None else '  N/A')
        mrr = (f"{m.get('mrr'):.4f}"      if m.get('mrr')      is not None else '  N/A')
        rk  = (f"{m['recall_at_K']:.4f}"  if m['recall_at_K']  is not None else '  N/A')
        r3k = (f"{m['recall_at_3K']:.4f}" if m['recall_at_3K'] is not None else '  N/A')
        print(f'{tag:<18} | {db_name:<22} | {pr:>8} | {mrr:>8} | {rk:>8} | {r3k:>8} | '
              f'{m["n_gt"]:>4} | {m["total_pairs"]:>7}')
    print('-' * 120)

# ── 4) 최종 macro / 순위 출력 ─────────────────────────────────
with open(RANKING_JSON_PATH) as f:
    final = json.load(f)

print('\n' + '=' * 80)
print(f'=== 평균 PR-AUC / Recall@K (config × DB 평균) ===')
print('=' * 80)
print(f'{"config":<18} | {"avg PR-AUC":>10} | {"avg MRR":>10} | {"avg R@K":>10} | {"avg R@3K":>10}')
print('-' * 95)
for entry in final['avg_summary']:
    note = '  ← v34 baseline (full)' if entry['tag'] == 'stat+char+ov' else ''
    pr  = (f"{entry['avg_pr_auc']:.4f}"       if entry['avg_pr_auc']       is not None else '  N/A')
    mrr = (f"{entry.get('avg_mrr'):.4f}"      if entry.get('avg_mrr')      is not None else '  N/A')
    rk  = (f"{entry['avg_recall_at_K']:.4f}"  if entry['avg_recall_at_K']  is not None else '  N/A')
    r3k = (f"{entry['avg_recall_at_3K']:.4f}" if entry['avg_recall_at_3K'] is not None else '  N/A')
    print(f'{entry["tag"]:<18} | {pr:>10} | {mrr:>10} | {rk:>10} | {r3k:>10}{note}')

print(f'\n💾 저장 위치: {RANKING_JSON_PATH}')
print(f'   → 통합 비교 노트북(30_compare_results.ipynb)에서 자동 탐지됨')


In [ ]:
# ── Classic metrics (TP/FP/FN/(TN), Precision/Recall/F1) 저장 ─────────
# ★ 재실행 정책: 셀 실행 시 results_table 전체로 덮어쓰기 (수동 업데이트) ★
# results_table (cell 42 에서 산출) 을 _classic 파일로 저장.
# 통합 노트북에서 ranking 지표와 함께 비교 가능.

import json
import os

classic_path = os.path.join(
    base_path,
    f'ablation_ranking_metrics_{post_fix}{RUN_SUFFIX}_classic.json'
)

classic_data = {
    'seed':      SEED,
    'threshold': 0.5,
    'results_table': {
        tag: {
            db: {
                'tp':        int(m.get('tp', 0)),
                'fp':        int(m.get('fp', 0)),
                'fn':        int(m.get('fn', 0)),
                'tn':        int(m.get('tn', 0)) if 'tn' in m else None,
                'precision': round(float(m.get('precision', 0.0)), 4),
                'recall':    round(float(m.get('recall', 0.0)), 4),
                'f1':        round(float(m.get('f1', 0.0)), 4),
            } for db, m in dbs.items()
        } for tag, dbs in results_table.items()
    },
}

# Avg F1 / Precision / Recall per config (4 DB 평균)
avg_classic = []
for tag in classic_data['results_table'].keys():
    dbs = classic_data['results_table'][tag]
    vals_f1 = [d['f1']        for d in dbs.values()]
    vals_p  = [d['precision'] for d in dbs.values()]
    vals_r  = [d['recall']    for d in dbs.values()]
    avg_classic.append({
        'tag':           tag,
        'avg_f1':        round(sum(vals_f1)/len(vals_f1), 4) if vals_f1 else None,
        'avg_precision': round(sum(vals_p)/len(vals_p), 4)   if vals_p  else None,
        'avg_recall':    round(sum(vals_r)/len(vals_r), 4)   if vals_r  else None,
    })
classic_data['avg_summary'] = avg_classic

# 마스킹/비마스킹은 서로 다른 파일로 분리 저장 (RUN_SUFFIX 자동 적용)
with open(classic_path, 'w') as f:
    json.dump(classic_data, f, indent=2)

print(f"[저장] {classic_path}")
print(f"   configs: {len(classic_data['results_table'])}, "
      f"DBs/config: {len(next(iter(classic_data['results_table'].values()), {}))}")

In [ ]:
# 🔍 진단 셀 — superhero vs superhero_mask 의 per-pair score 비교
# cell 41 의 result 객체를 활용. 두 폴더에서 각각 infer 한 후 비교.

import pandas as pd
import os

# 1) 두 폴더 결과를 각각 infer
results_by_db = {}
for db_name_diag in ['superhero', 'superhero_mask']:
    apply_seed(SEED)  # 결정론적 비교 위해
    test_folder_diag = f'{TEST_ROOT}/{db_name_diag}'
    result_diag = infer_folder(
        test_folder_diag, model, extractor, scaler, ov_scaler,
        device=device, threshold=0.5, apply_hard_constraint=True, verbose=False,
    )
    results_by_db[db_name_diag] = result_diag

# 2) 두 결과의 (col_a, col_b) score 를 logical 컬럼명으로 매핑
import json
mask_map_path = f'{TEST_ROOT}/superhero_mask/mask_mapping.json'
mm = json.load(open(mask_map_path))
table_map = dict(mm.get('table_mapping', []))   # orig_table -> mask_table
col_map_all = dict(mm.get('col_mapping', []))   # (orig_table, orig_col) → (mask_table, mask_col)

# 3) score 차이 분석
orig_scores = {}
for r in results_by_db['superhero']['all_results']:
    key = frozenset({(r['A_table'], r['A_col']), (r['B_table'], r['B_col'])})
    orig_scores[key] = r['FK_Score']

mask_scores = {}
for r in results_by_db['superhero_mask']['all_results']:
    key = frozenset({(r['A_table'], r['A_col']), (r['B_table'], r['B_col'])})
    mask_scores[key] = r['FK_Score']

# logical 컬럼명 (orig) 로 mask key 를 변환해서 비교
# col_mapping 구조 확인 필요 → 사용자가 출력 보고 적절히 조정 가능
print(f"orig 결과 쌍 수: {len(orig_scores)}")
print(f"mask 결과 쌍 수: {len(mask_scores)}")
print(f"mask_mapping 구조: {list(mm.keys())}")
print(f"col_mapping 샘플: {list(col_map_all.items())[:3] if isinstance(col_map_all, dict) else col_map_all[:3]}")

# 4) score 분포 차이 통계
import numpy as np
orig_score_arr = sorted(orig_scores.values(), reverse=True)
mask_score_arr = sorted(mask_scores.values(), reverse=True)
print(f"\norig score top 5: {orig_score_arr[:5]}")
print(f"mask score top 5: {mask_score_arr[:5]}")
print(f"orig score mean: {np.mean(orig_score_arr):.4f}, std: {np.std(orig_score_arr):.4f}")
print(f"mask score mean: {np.mean(mask_score_arr):.4f}, std: {np.std(mask_score_arr):.4f}")


In [ ]:
# 🔍 정밀 진단 — col_mapping 으로 logical pair 매칭하여 score 차이 검사

import json
import numpy as np

# mask_mapping 로드
mm = json.load(open(f'{TEST_ROOT}/superhero_mask/mask_mapping.json'))
table_map = dict(mm['table_mapping'])           # orig_table -> mask_table
col_map_per_table = dict(mm['col_mapping'])     # orig_table -> {orig_col -> mask_col}

# orig (table, col) -> mask (table, col) 전체 매핑 구축
o2m_col = {}  # (orig_t, orig_c) -> (mask_t, mask_c)
for ot, col_dict in col_map_per_table.items():
    mt = table_map[ot]
    for oc, mc in col_dict.items():
        o2m_col[(ot, oc)] = (mt, mc)

# orig 결과의 각 pair 를 mask key 로 변환해서 mask_scores 에서 lookup
diffs = []
for r_orig in results_by_db['superhero']['all_results']:
    o_a = (r_orig['A_table'], r_orig['A_col'])
    o_b = (r_orig['B_table'], r_orig['B_col'])
    if o_a not in o2m_col or o_b not in o2m_col:
        continue
    m_a = o2m_col[o_a]
    m_b = o2m_col[o_b]

    # mask_scores 는 frozenset 키
    mask_key = frozenset({m_a, m_b})
    s_mask = mask_scores.get(mask_key)
    s_orig = r_orig['FK_Score']

    if s_mask is None:
        continue

    diff = s_orig - s_mask
    diffs.append({
        'orig_pair': f"{o_a[0]}.{o_a[1]} ↔ {o_b[0]}.{o_b[1]}",
        'mask_pair': f"{m_a[0]}.{m_a[1]} ↔ {m_b[0]}.{m_b[1]}",
        'orig_score': s_orig,
        'mask_score': s_mask,
        'diff': diff,
        'abs_diff': abs(diff),
    })

# 차이 큰 순으로 정렬
diffs.sort(key=lambda x: -x['abs_diff'])

n_match = sum(1 for d in diffs if d['abs_diff'] < 1e-6)
n_diff  = sum(1 for d in diffs if d['abs_diff'] >= 1e-6)
print(f"매칭된 pair: {len(diffs)}개")
print(f"  score 일치: {n_match}개")
print(f"  score 다름: {n_diff}개")
print()

if n_diff > 0:
    print(f"=== score 차이 큰 top 10 pair ===")
    print(f"{'orig_pair':<70} {'orig':>8} {'mask':>8} {'diff':>8}")
    print("-" * 100)
    for d in diffs[:10]:
        print(f"{d['orig_pair']:<70} {d['orig_score']:>8.4f} {d['mask_score']:>8.4f} {d['diff']:>+8.4f}")

    # 차이의 크기 분포
    abs_diffs = [d['abs_diff'] for d in diffs if d['abs_diff'] >= 1e-6]
    print(f"\n=== 차이 분포 (n={len(abs_diffs)}) ===")
    print(f"  mean abs diff : {np.mean(abs_diffs):.6f}")
    print(f"  max  abs diff : {np.max(abs_diffs):.6f}")
    print(f"  median diff   : {np.median(abs_diffs):.6f}")

    # 가장 큰 차이 pair 의 실제 데이터 비교
    worst = diffs[0]
    print(f"\n=== 가장 큰 차이 pair 의 데이터 직접 비교 ===")
    print(f"  orig: {worst['orig_pair']}")
    print(f"  mask: {worst['mask_pair']}")
